**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/catalystneuro/encore", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/talbot-2018"], check=True)
    os.chdir("/content/encore/entries/talbot-2018")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Talbot et al. (2018) on CRCNS hc-16: single cells, coordination and frame-specific firing

Paper: Talbot, Sparks, Dvorak, Curran, Alarcon and Fenton (2018), "Normal CA1 Place Fields but
Discoordinated Network Discharge in a Fmr1-Null Mouse Model of Fragile X Syndrome", Neuron
97:684-697, doi:10.1016/j.neuron.2017.12.043.

Dataset: CRCNS hc-16 (Sparks, Talbot, Dvorak and Fenton), doi:10.6080/K01V5C44, read from the
public bucket `crcnsarchive` at `hc-16/live/data.zip`, streamed with fsspec and zipfile.

Version: CRCNS data sets are not versioned, so the version is recorded as `unversioned`. The
paper cites the data set's DOI only (Key Resources Table), and the bucket holds one copy of the
data, under `live/` (there is no `old/`), so that copy is the one read. The cell below reads the
bucket's `checksums.md5`; the file read is `data.zip`, MD5 45b534dd6dbeb94db539bc429223eecb
(data descriptor version 0.6, April 26, 2018). A later change of the zip would change that sum.

Claims: `claims/crcns-hc-16/10.1016-j.neuron.2017.12.043.yaml`, written from the version of
record and its supplement (Figures S1 to S6, Tables S1 and S2), as its opening comment says.

| Claim | Here |
|---|---|
| fig1ab-baseline-transmission, fig1cd-potentiation, figs1-avoidance-equal | blocked: slice recordings and slice-cohort behaviour not in the data set |
| unit-and-cell-counts | blocked: the data set holds 15 of the paper's 21 mice and no isolation measures (counts printed for the record) |
| interneuron-proportion | tested (rate and bursting only; no waveforms) |
| place-cell-proportion | tested |
| figs3-nonspatial-properties-similar | tested (no waveform width) |
| fig2-spatial-quality-similar | tested (large box only) |
| tables1-place-cell-properties-similar | tested |
| tables1-nonspatial-active-pixels | tested |
| figs4-remapping-similar | blocked: no box and cylinder recordings of the same cells (shown below) |
| fig3b-place-cell-modulation-similar, fig3bc-weaker-modulation-nonspatial-interneurons, fig3b-phase-preference, fig3d-stereotypy, figs5a-power-spectra-similar | blocked: no LFP in the data set |
| overdispersion-passes, overdispersion-cells | tested |
| fig4b-covariance-30-passes, fig4b-covariance-40-passes | tested |
| fig4cd-kendall-place-cells, fig4d-network-state-recurrence, kendall-nonspatial-pyramidal | tested |
| kendall-quiet-states | tested (low-speed periods; no sleep recordings) |
| locomotion-similar | tested |
| fig5b-frame-specific-ipos, fig5c-ensemble-coordination, figs6-room-frame-preference | tested |
| fig6-covariance-sem-predicts-errors, fig6-kendall-predicts-errors, discussion-single-cell-measures-no-relation, fig6-avoidance-behavior | blocked: avoidance sessions not in the data set |

Population used throughout: the data set has no isolation measures (IsoIBG, IsoINN) and no
waveforms, so the paper's well-isolated cells cannot be selected. Every cluster other than
cluster 0 of a tetrode is used. This is the largest difference from the paper's method and bears
on every tested claim. Two variants bear on it: a refractory filter (clusters with at most 0.5%
of interspike intervals below 2 ms), a proxy for isolation, and single-cell measures on the code's
32 x 32 grid, which needs no spatial scale.

In [ ]:
import sys, time, hashlib, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import hc16

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
T0 = time.time()
md5 = hc16.check_md5_list()
print("checksums.md5:", md5)
assert md5["data.zip"] == hc16.DATA_ZIP_MD5
z = hc16.open_zip()
files = hc16.list_members(z)
sessions = hc16.load_sessions(files)
S = {s.id: s for s in sessions}
print(f"loaded {len(sessions)} sessions in {time.time() - T0:.0f} s")

In [ ]:
print("Zip members matched by name:", len(files), "of", sum(1 for i in z.infolist() if not i.is_dir()),
      "(the rest are .DS_Store)")
print(files.groupby(["env", "genotype", "frame", "part", "times"]).size().rename("files").to_string())

sess = pd.DataFrame([dict(session=s.id, env=s.env, genotype=s.genotype, mouse=s.mouse, date=s.date,
                          cond=s.cond, cut=s.cut, rate_hz=round(s.rate, 3), duration_s=round(s.duration, 1),
                          frames="+".join(sorted(s.frames)), clusters=len(s.counts), cells=len(s.cells),
                          untracked=round(1 - s.tracked.mean(), 3),
                          extent_px=f"{s.extent_px[0]:.0f}x{s.extent_px[1]:.0f}",
                          px_per_cm=round(s.px_per_cm, 2), leftover_bytes=s.leftover)
                     for s in sessions])
print(sess.to_string())
print("\nOne full TS header (%s, first file):" % sessions[0].id)
print(sessions[0].header)
print("Header fields over every file read:", sorted(set().union(*[s.header_fields for s in sessions])))
print("SCALE_X and SCALE_Y values in the first file of each session:",
      sorted({v for s in sessions for v in re.findall(r"%SCALE_[XY]\S*\s+([\d.]+)", s.header)}))
print("\nMice per genotype:", sess.groupby("genotype").mouse.nunique().to_dict())
print("Sessions per environment and genotype:\n", sess.groupby(["env", "genotype"]).size().unstack())
print("Position sampling rate (Hz): %.3f to %.3f; durations %.0f to %.0f s"
      % (sess.rate_hz.min(), sess.rate_hz.max(), sess.duration_s.min(), sess.duration_s.max()))

# cluster 0 of each tetrode
c0 = [(s.id, u, s.counts[u].sum() / s.duration) for s in sessions for u in s.counts if u.endswith("C0")]
c0r = np.array([r for _, _, r in c0])
print(f"\nCluster 0: {len(c0)} files, rate median {np.median(c0r):.1f} AP/s (range {c0r.min():.1f} to {c0r.max():.1f})")
allr = np.array([s.counts[u].sum() / s.duration for s in sessions for u in s.cells])
print(f"Other clusters: {len(allr)}, rate median {np.median(allr):.2f} AP/s (range {allr.min():.3f} to {allr.max():.1f})")

# spikes are sorted, inside the session, and each lies near its position sample
bad_sort = 0; offs = []
for s in sessions:
    for u in s.cells:
        sp = s.spikes[u]
        bad_sort += int(np.any(np.diff(sp) < 0))
        k = np.repeat(np.arange(s.nsamp), s.counts[u])
        offs.append(((sp - s.t0) / 1e4 * s.rate - k))
offs = np.concatenate(offs)
print(f"Clusters with unsorted spike times: {bad_sort}")
print("Spike time minus the nominal time of its position sample (t0 + k/rate), in samples: "
      "percentiles 0/50/100 = %.2f / %.2f / %.2f" % tuple(np.percentile(offs, [0, 50, 100])))
res = []
for s in sessions:
    for u in s.counts:
        k = np.repeat(np.arange(s.nsamp), s.counts[u])
        res.append((s.spikes[u] / 1e4 - s.sample_time[k]) * s.rate)
res = np.concatenate(res)
print("Spike time minus the estimated time of its sample (hc16.Session.sample_time), in samples: "
      "percentiles 0.1/50/99.9 = %.2f / %.2f / %.2f" % tuple(np.percentile(res, [0.1, 50, 99.9])))
for s in sessions:
    for u in s.cells:
        d = np.diff(s.spikes[u])
        if np.any(d < 0):
            print(f"  unsorted: {s.id} {u}: {int((d < 0).sum())} backward steps of {len(d) + 1} spikes, "
                  f"largest {-d.min() / 10:.1f} ms")

# x and y range (tracker pixels, 0 to 255; (0, 0) is untracked)
for env, g in sess.groupby("env"):
    print(f"{env}: room-frame extent {', '.join(g.extent_px)} px; px/cm {g.px_per_cm.min()} to {g.px_per_cm.max()}")

Checks of the data against the description, and of the files against each other:

- The descriptor gives x and y in 0-255 pixels and times in 100 us units, but no spatial scale.
  The scale is taken from the extent of the tracked room-frame positions (0.1 to 99.9
  percentiles) and the enclosure sizes of the STAR Methods (large box 50 cm, arena 40 cm). This is
  an assumption; the printed px/cm differ between box sessions (about 3.6 and 4.4), which fits
  two camera set-ups and is consistent with, not proof of, the assumed sizes.
- The TS headers hold the sampling rate, the first timestamp and a track-to-map scale of 1.00;
  no isolation measure and no spatial scale in cm (printed above).
- The descriptor's example header has a sampling interval of 58.3 samples/s; every file here has
  30.0, matching the 30 frames/s of the STAR Methods.
- The header's sampling rate and first timestamp do not give the times of the position samples:
  spikes lie up to 118 samples (about 4 s) after the nominal time t0 + k/rate of the sample they
  are filed under, as the rate drifts against the spike clock and frames are dropped. The helper
  estimates each sample's time from the spikes filed under it; after that the spikes lie 0 to 1
  sample after their sample's time, as they should. Six spike trains have one backward step in
  their times (printed above).
- The next cells check that the `.times` files repeat the spike times of the TS files, that the
  `_0-15min` and `_15-30min` files are parts of the full sessions (one position sample at the
  split is in neither part, so a spike filed under it is lost), that the arena-frame files of
  stationary sessions repeat the room frame, and whether any recording is deposited twice.

In [ ]:
rng = np.random.default_rng(0)
# .times files against the TS spikes, for every cell of three sessions
chk = []
for sid in ["M16_090414_stationary1", "M34_041015_blackbox", "M37_051115_rotating"]:
    s = S[sid]
    for u in s.cells:
        nm = files[(files.mouse == s.mouse) & (files.date == s.date) & (files.cond == s.cond) & (files.unit == u)
                   & (files.frame == "Rm") & (files.part == "full") & files.times].name.iloc[0]
        v = np.array(z.read(nm).split(), dtype=np.int64)
        chk.append(np.array_equal(v, s.spikes[u]))
print(f".times files identical to the TS spike times: {sum(chk)} of {len(chk)} cells (3 sessions)")

# 0-15 and 15-30 min parts against the full session
s = S["M16_090414_rotating"]
part = hc16.read_members([f"data/rotating_cylinder/WT/M16_090414_rotating-cl101015-fix_0-15min.{u}Rm" for u in s.cells[:5]]
                         + [f"data/rotating_cylinder/WT/M16_090414_rotating-cl101015-fix_15-30min.{u}Rm" for u in s.cells[:5]])
for u in s.cells[:5]:
    a = part[f"data/rotating_cylinder/WT/M16_090414_rotating-cl101015-fix_0-15min.{u}Rm"]
    b = part[f"data/rotating_cylinder/WT/M16_090414_rotating-cl101015-fix_15-30min.{u}Rm"]
    joined = np.r_[a["spk"], b["spk"]]
    print(f"{u}: 0-15 min {len(a['x'])} samples, 15-30 min {len(b['x'])}, full {s.nsamp}; "
          f"spikes {len(a['spk'])}+{len(b['spk'])} vs {len(s.spikes[u])}, "
          f"parts equal to the full spike train: {np.array_equal(joined, s.spikes[u])}")
na, nb_ = len(a["x"]), len(b["x"])
print("First part equals the full file's first samples:", np.array_equal(a["x"], s.frames["Rm"][0][:na]),
      "; second part equals its last samples:", np.array_equal(b["x"], s.frames["Rm"][0][s.nsamp - nb_:]))
dropped = np.arange(na, s.nsamp - nb_)
print("Samples of the full file in neither part:", dropped.tolist(),
      "; spikes there per cell:", {u: int(s.counts[u][dropped].sum()) for u in s.cells[:5]})

# arena frame of stationary sessions
for s in sessions:
    if "Ar" in s.frames and s.env == "stationary_cylinder":
        print(f"{s.id}: arena-frame positions identical to room frame: "
              f"{np.array_equal(s.frames['Ar'][0], s.frames['Rm'][0]) and np.array_equal(s.frames['Ar'][1], s.frames['Rm'][1])}")
for s in sessions:
    if s.env == "rotating_cylinder":
        same = np.mean((s.frames["Ar"][0] == s.frames["Rm"][0]) & (s.frames["Ar"][1] == s.frames["Rm"][1]))
        print(f"{s.id}: fraction of samples with equal room and arena positions {same:.3f}")

# recordings deposited twice: identical spike trains or identical position series across sessions
h_spk, h_pos = {}, {}
for s in sessions:
    h_pos.setdefault(hashlib.md5(s.frames["Rm"][0].tobytes() + s.frames["Rm"][1].tobytes()).hexdigest(), []).append(s.id)
    for u in s.cells:
        if len(s.spikes[u]) > 10:
            h_spk.setdefault(hashlib.md5(s.spikes[u].tobytes()).hexdigest(), []).append(f"{s.id}.{u}")
dup_pos = [v for v in h_pos.values() if len(v) > 1]
dup_spk = [v for v in h_spk.values() if len(v) > 1]
dup_spk_x = [v for v in dup_spk if len({x.split('.')[0] for x in v}) > 1]
print(f"Position series shared by more than one session: {dup_pos}")
print(f"Identical spike trains across sessions: {len(dup_spk_x)}; within a session: {len(dup_spk) - len(dup_spk_x)}")
print(dup_spk[:10])

Clock check. Spikes and positions are in the same file, each spike filed under a position
sample. To show that spike counts and positions share a clock, the spatial information of every
cluster is computed with the true alignment and with the spike counts shifted in time against
the positions; if the alignment were off, the true alignment would not stand out.

In [ ]:
def info_shifted(s, shift_s):
    idx, shape = hc16.pixel_index(s, "Rm")
    occ = hc16.occupancy(s, idx, shape)
    k = int(round(shift_s * s.rate))
    out = []
    for u in s.cells:
        n = s.counts[u]
        s.counts[u] = np.roll(n, k)
        rm, o = hc16.rate_map(s, u, idx, shape, occ)
        s.counts[u] = n
        out.append(hc16.information(rm, o))
    return np.array(out)

shifts = [0, 0.5, 2, 10, 60, 300]
rows = []
for s in sessions:
    if s.env == "rotating_cylinder":
        continue
    for sh in shifts:
        v = info_shifted(s, sh)
        rows.append(dict(session=s.id, shift_s=sh, median_info=np.nanmedian(v)))
clk = pd.DataFrame(rows).pivot(index="session", columns="shift_s", values="median_info")
print("Median information content (bits/spike) of each session's clusters, by shift of spikes against positions:")
print(clk.round(3).to_string())
print("Sessions where the true alignment gives the highest median:", int((clk.idxmax(axis=1) == 0).sum()), "of", len(clk))

In 29 of 34 stationary sessions the true alignment gives the highest median information content,
and the median falls with the shift in most of the rest; this is consistent with spikes and
positions sharing a clock, not proof of it.

## Cell table and classification

One row per cluster and session, with the single-cell measures on unsmoothed 1.25 cm room-frame
maps. Classes (STAR Methods, "Classification of Cell Types"; the waveform criterion cannot be
applied): interneuron, rate above 2 AP/s; pyramidal, rate below 2 AP/s with an ISI histogram peak
below 10 ms; place cell, a pyramidal cell with rate 0.1-2 AP/s, coherence above 0.4 and
information above 0.4 bits/spike ("Characterization of Place Cells"). Clusters below 2 AP/s
without bursts are left unclassified. A variant drops the burst criterion (rate alone).

"Standard recordings" (the session a cell is classified in) is open in the paper; the claims
file names two readings. R1: each cell once, from its first stationary-environment session (the
large box, or the first stationary arena session of a day). R2: every stationary session (large
box, stationary arena 1 and 2) on its own, a cell counted per session. Sessions of a day share a
cluster cutting file (the `cl...` part of the name), so a cluster name within a cut is taken to be
the same cell across that day's sessions (an assumption: the files carry no other identifier).

In [ ]:
cells = hc16.cell_table(sessions)
cells["cls"] = hc16.classify(cells)
cells["cls_rate"] = hc16.classify(cells, use_burst=False)
stationary = cells[cells.env != "rotating_cylinder"].copy()
first_sid = {}
for s in sessions:
    if s.env == "rotating_cylinder":
        continue
    first_sid.setdefault(f"{s.mouse}_{s.date}", s.id)  # sessions are sorted: stationary1 before stationary2
R1 = stationary[stationary.session.isin(set(first_sid.values()))].copy()
R2 = stationary.copy()
print("R1 sessions:", sorted(first_sid.values()))
for name, t in [("R1", R1), ("R2", R2)]:
    print(f"\n{name}: classes by genotype (rate and bursting)")
    print(t.groupby(["genotype", "cls"]).size().unstack(fill_value=0))
    print(f"{name}: classes by genotype (rate only)")
    print(t.groupby(["genotype", "cls_rate"]).size().unstack(fill_value=0))
print("\nRanges of the measures used (all stationary sessions):")
print(R2[["rate", "coherence", "info", "active", "burst_prob", "isi_mean", "isi_peak_ms", "infield_mean", "infield_20pk"]]
      .describe().round(3).to_string())

Two variants of the population and of the maps, shown with the claims they bear on. The data
set gives no isolation measure, so the fraction of interspike intervals below 2 ms (refractory
violations) is shown as a proxy: a cluster mixing several units has many. The refractory filter
keeps clusters with at most 0.5% of intervals below 2 ms, a standard choice. The spatial scale is
an assumption, so the single-cell measures are also computed on the code's 32 x 32 grid of the
256 x 256 tracker frame (8 tracker pixels a side), which needs no scale.

In [ ]:
print("Fraction of ISIs below 2 ms by class, all sessions:")
print(cells.groupby("cls").refractory.describe(percentiles=[0.5, 0.9]).round(4).to_string())
ISO = 0.005
print(f"Clusters with more than {ISO:.1%} of ISIs below 2 ms:", int((cells.refractory > ISO).sum()), "of", len(cells))
cells_code = hc16.cell_table(sessions, index_fn=hc16.pixel_index_code)
cells_code["cls"] = hc16.classify(cells_code)
print("32 x 32 grid pixel side in cm at the assumed scale:",
      {env: f"{8 / g.px_per_cm.max():.2f} to {8 / g.px_per_cm.min():.2f}" for env, g in sess.groupby("env")})
print("\nClass on 1.25 cm pixels (rows) against the 32 x 32 grid (columns), all sessions:")
print(pd.crosstab(cells.cls, cells_code.cls).to_string())
R1_code = cells_code.loc[R1.index]; R2_code = cells_code.loc[R2.index]
R1_iso = R1[R1.refractory <= ISO]; R2_iso = R2[R2.refractory <= ISO]
for name, t in [("R1, 32 x 32 grid", R1_code), ("R2, 32 x 32 grid", R2_code),
                ("R1, refractory filter", R1_iso), ("R2, refractory filter", R2_iso)]:
    print(f"\n{name}: classes by genotype")
    print(t.groupby(["genotype", "cls"]).size().unstack(fill_value=0).to_string())
print("\nInterneuron rate (AP/s), R1: quantiles by genotype")
print(R1[R1.cls == "interneuron"].groupby("genotype").rate.describe(percentiles=[0.25, 0.5, 0.75]).round(2).to_string())

## unit-and-cell-counts

> Of 1,115 clusters, 499 came from 12 WT and 616 from 9 Fmr1-null mice; 226 WT and 334
> Fmr1-null well-isolated cells formed the data set.

Criterion: mice counted exactly (11 or 12 WT); clusters and cells counted once or per session.
The data set holds no isolation measures, so the well-isolated cells cannot be counted; the
cluster counts are given with and without cluster 0.

In [ ]:
mice = sess.groupby("genotype").mouse.unique()
print("Mice in the data set:", {g: sorted(v) for g, v in mice.items()})
print("Mice per genotype:", sess.groupby("genotype").mouse.nunique().to_dict(), "(paper: WT 12, Fmr1-null 9)")
cnt = []
for s in sessions:
    cnt.append(dict(session=s.id, genotype=s.genotype, env=s.env, mouse=s.mouse, day=f"{s.mouse}_{s.date}",
                    cut=s.cut, clusters=len(s.cells), with_c0=len(s.counts)))
cnt = pd.DataFrame(cnt)
# once: each day's cut counted once (a cluster name within a cut is one cell across the day's sessions)
once = cnt.drop_duplicates(["day", "cut"])
once_day = cnt.sort_values("session").drop_duplicates(["day"])
tab = pd.DataFrame({
    "once per cut file": once.groupby("genotype").clusters.sum(),
    "once per day (R1)": R1.groupby("genotype").size(),
    "per session, stationary (R2)": R2.groupby("genotype").size(),
    "per session, all sessions": cnt.groupby("genotype").clusters.sum(),
    "once per cut, with cluster 0": once.groupby("genotype").with_c0.sum(),
    "per session, all, with cluster 0": cnt.groupby("genotype").with_c0.sum(),
})
print("\nClusters (cluster 0 left out unless said):")
print(tab.T.to_string())
print("Paper: clusters WT 499, Fmr1-null 616; well-isolated cells WT 226, Fmr1-null 334 (tolerance 10%)")
for col in tab.columns:
    w, k = tab.loc["WT", col], tab.loc["Fmr1-null", col]
    print(f"  {col}: WT {w} ({'in' if abs(w - 499) <= 49.9 else 'out of'} 499 +/- 49.9), "
          f"Fmr1-null {k} ({'in' if abs(k - 616) <= 61.6 else 'out of'} 616 +/- 61.6); "
          f"as cells: WT {'in' if abs(w - 226) <= 22.6 else 'out of'} 226 +/- 22.6, "
          f"Fmr1-null {'in' if abs(k - 334) <= 33.4 else 'out of'} 334 +/- 33.4")

**Result: blocked (data missing).** The data set holds 8 WT and 7 Fmr1-null mice, where the paper
has 12 (or 11) and 9: it is a subset of the paper's data, so its counts cannot test the paper's.
The well-isolated cells cannot be selected either, as the data set has no isolation measures. For
the record, the cluster counts are within 10% of 499 and 616 only when every session is counted
on its own and cluster 0 (the unsorted spikes of each tetrode) is included, 490 WT and 557
Fmr1-null, which with 15 of 21 mice is a coincidence; counted once per cut file without cluster 0
there are 203 WT and 294 Fmr1-null clusters.

## figs4-remapping-similar (blocked)

The claim compares the same cells in a square box and the circular arena. The table pairs each
mouse's box sessions with its arena sessions, with their dates and cut files.

In [ ]:
rows = []
for m, g in sess.groupby("mouse"):
    for _, b in g[g.env == "big_box"].iterrows():
        for _, a in g[g.env != "big_box"].iterrows():
            rows.append(dict(mouse=m, genotype=b.genotype, box=b.session, box_cut=b.cut, arena=a.session,
                             arena_cut=a.cut, same_day=b.date == a.date, same_cut=b.cut == a.cut))
fx = pd.DataFrame(rows)
print(fx.to_string())
print("Box and arena session pairs:", len(fx), "; on the same day:", int(fx.same_day.sum()),
      "; with the same cut file:", int(fx.same_cut.sum()))
print("Small box sessions in the data set:", int((files.env == "small_box").sum()))

No box session shares a day or a cut file with an arena session of the same mouse, and the data
set has no waveforms, so no cell can be followed from the square to the circular enclosure. The
claim is blocked (data missing): the small box sessions recorded on the days of arena sessions
(Table S2, for example M32 on 04-17-15) are not in the data set.

## interneuron-proportion

> A lower proportion of the well-isolated cells is classified as interneurons in WT (28, 12.4%)
> than in Fmr1-null mice (98, 29.3%).

Criterion: percentages within tolerance (12.4 +/- 1.24, 29.3 +/- 2.93) and a two-proportion z test
with p < 0.05, WT lower. Classification by rate and bursting, the variant the claims file names
when waveforms are missing; rate alone is shown too. Readings R1 and R2 for the session.

In [ ]:
def proportions(t, col, cls, denom_classes=None):
    out = {}
    for g in ["WT", "Fmr1-null"]:
        tt = t[t.genotype == g]
        d = tt if denom_classes is None else tt[tt[col].isin(denom_classes)]
        out[g] = (int((tt[col] == cls).sum()), len(d))
    z, p = hc16.prop_test(out["WT"][0], out["WT"][1], out["Fmr1-null"][0], out["Fmr1-null"][1])
    return out, z, p

rows = []
for rname, t in [("R1", R1), ("R2", R2), ("R1 refractory filter", R1_iso), ("R2 refractory filter", R2_iso)]:
    for col in ["cls", "cls_rate"]:
        for den_name, den in [("all clusters", None), ("classified cells", ["interneuron", "place", "nonspatial"])]:
            if col == "cls_rate" and den is not None:
                continue
            o, zz, pp = proportions(t, col, "interneuron", den)
            rows.append(dict(reading=rname, classes="rate+burst" if col == "cls" else "rate only",
                             denominator=den_name, WT=f"{o['WT'][0]}/{o['WT'][1]}",
                             WT_pct=100 * o["WT"][0] / o["WT"][1], KO=f"{o['Fmr1-null'][0]}/{o['Fmr1-null'][1]}",
                             KO_pct=100 * o["Fmr1-null"][0] / o["Fmr1-null"][1], z=zz, p_two_sided=pp))
ip = pd.DataFrame(rows)
ip["WT_in_tol"] = (ip.WT_pct - 12.4).abs() <= 1.24
ip["KO_in_tol"] = (ip.KO_pct - 29.3).abs() <= 2.93
print(ip.round(4).to_string())
print("\nBy mouse (R1, rate+burst, all clusters): interneuron fraction")
print(R1.groupby(["genotype", "mouse"]).apply(lambda d: pd.Series(dict(n=len(d), interneurons=(d.cls == "interneuron").sum(),
      frac=(d.cls == "interneuron").mean()))).round(3).to_string())

**Result: partially reproduced.** Fmr1-null has the larger share of interneurons, as claimed,
under every reading: R1, all clusters, 44.5% against 33.5% (z = -2.36, p = 0.018); R2, 48.5%
against 35.4% (z = -3.33, p = 0.0009); with the refractory filter p = 0.092 (R1) and 0.023 (R2).
The percentages are far from the paper's under every reading: WT 33.0% to 41.4% against
12.4 +/- 1.24%, Fmr1-null 41.0% to 55.7% against 29.3 +/- 2.93%. Interneurons here are clusters
above 2 AP/s, without the waveform criterion, among every sorted cluster. Few clusters have
refractory violations (70 of 774 have more than 0.5% of intervals below 2 ms), and the filter
changes the percentages by at most 4.9 points, so this filter does not remove the excess. It is
a weak proxy for the paper's isolation measures: interneuron rates are still 4.34 (WT) and 6.95
(Fmr1-null) AP/s at the median in R1, so whether poorly isolated clusters explain the excess
cannot be told from these data.

## place-cell-proportion

> Place cells are less prevalent in Fmr1-null mice (141, 42.2%) than in WT mice (108, 47.8%).

Criterion: counts and percentages within tolerance under one denominator (all cells, or the
principal cells), and a two-proportion z test with p < 0.05 (one-sided; two-sided as a variant),
Fmr1-null lower.

In [ ]:
rows = []
for rname, t in [("R1", R1), ("R2", R2), ("R1 32x32", R1_code), ("R2 32x32", R2_code),
                  ("R1 refractory filter", R1_iso), ("R2 refractory filter", R2_iso)]:
    for col in [c for c in ["cls", "cls_rate"] if c in t]:
        for den_name, den in [("all clusters", None), ("classified cells", ["interneuron", "place", "nonspatial"]),
                              ("principal cells", ["place", "nonspatial"])]:
            if col == "cls_rate" and den == ["interneuron", "place", "nonspatial"]:
                continue
            o, zz, pp = proportions(t, col, "place", den)
            rows.append(dict(reading=rname, classes="rate+burst" if col == "cls" else "rate only",
                             denominator=den_name, WT=o["WT"][0], WT_n=o["WT"][1], WT_pct=100 * o["WT"][0] / o["WT"][1],
                             KO=o["Fmr1-null"][0], KO_n=o["Fmr1-null"][1],
                             KO_pct=100 * o["Fmr1-null"][0] / o["Fmr1-null"][1], z=zz,
                             p_one_sided=stats.norm.sf(zz), p_two_sided=pp))
pc = pd.DataFrame(rows)
pc["counts_in_tol"] = ((pc.WT - 108).abs() <= 10.8) & ((pc.KO - 141).abs() <= 14.1)
pc["pct_in_tol"] = ((pc.WT_pct - 47.8).abs() <= 4.78) & ((pc.KO_pct - 42.2).abs() <= 4.22)
print(pc.round(4).to_string())
print("\nBy mouse (R1, rate+burst): place cells / all clusters")
print(R1.groupby(["genotype", "mouse"]).apply(lambda d: f"{(d.cls == 'place').sum()}/{len(d)}").to_string())

**Result: not reproduced.** No reading gives the paper's counts or percentages: with all
clusters as the denominator there are 48 WT (23.6%) and 51 Fmr1-null (21.4%) place cells in R1,
and 55 (18.9%) and 70 (20.5%) in R2, against 108 (47.8%) and 141 (42.2%); with principal cells as
the denominator WT is 44.4% and Fmr1-null 51.5% (R1). No one-sided test over the 26 readings,
including the 32 x 32 grid and the refractory filter, finds Fmr1-null lower (smallest one-sided
p 0.27); with principal cells in R2 Fmr1-null is higher (53.0% against 37.7%, two-sided
p = 0.010; on the 32 x 32 grid 61.4% against 44.5%, p = 0.005).

## figs3-nonspatial-properties-similar

> Waveform width, burst probability, interspike interval and overall firing rate do not differ
> between genotypes.

Criterion: a two-way genotype x cell class ANOVA per measure (Figure S3 legend); burst
probability is the fraction of ISIs below 10 ms; ISI as the mean per cell (median as a variant);
reproduced when no genotype effect has p < 0.05. Waveforms are not in the data set, so width is
not tested. Cells: the classified clusters (place, non-spatial pyramidal, interneuron) of R1; R2
shown as a variant.

In [ ]:
def anova_rows(t, measures, label):
    rows = []
    for m in measures:
        a = hc16.anova2(t, m)
        rows.append(dict(reading=label, measure=m, n=int(t[m].notna().sum()),
                         F_genotype=a.loc["C(genotype)", "F"], df_resid=int(a.loc["Residual", "df"]),
                         p_genotype=a.loc["C(genotype)", "PR(>F)"], eta_p2_genotype=a.loc["C(genotype)", "eta_p2"],
                         p_class=a.loc["C(cls)", "PR(>F)"], p_interaction=a.loc["C(genotype):C(cls)", "PR(>F)"]))
    return rows

rows = []
for label, t in [("R1", R1), ("R2", R2), ("R1 refractory filter", R1_iso), ("R2 refractory filter", R2_iso)]:
    tc = t[t.cls.isin(["place", "nonspatial", "interneuron"])].copy()
    rows += anova_rows(tc, ["burst_prob", "isi_mean", "isi_median", "rate"], label)
f3 = pd.DataFrame(rows)
print(f3.round(4).to_string())
tc = R1[R1.cls.isin(["place", "nonspatial", "interneuron"])]
print("\nR1 means by genotype and class:")
print(tc.groupby(["cls", "genotype"])[["burst_prob", "isi_mean", "isi_median", "rate"]].agg(["mean", "count"]).round(3).to_string())
print("\nR1 by mouse: mean burst probability and rate of classified clusters")
print(tc.groupby(["genotype", "mouse"])[["burst_prob", "rate"]].mean().round(3).to_string())

**Result: partially reproduced.** In R1 (381 classified clusters) burst probability and the ISI
show no genotype effect (burst p = 0.13; mean ISI p = 0.40; median ISI p = 0.63), but the rate
does (F = 8.28, p = 0.0042, partial eta squared 0.022, with an interaction p = 0.0086; Fmr1-null
interneurons fire at 11.5 against 7.3 AP/s). In R2 burst probability also differs (p = 0.029) and
the rate (p = 0.0005). With the refractory filter the rate effect falls below significance
(R1 p = 0.070, R2 p = 0.057), so that in R1 no measure differs, but in R2 burst probability does
(p = 0.0037). Waveform width is not in the data set and is not tested.

## fig2-spatial-quality-similar

> In a fixed environment, spatial firing quality (coherence, proportion of active pixels,
> information content, number of place fields, in-field rate) differs between place cells and
> interneurons but not between genotypes.

Criterion: a two-way genotype x cell class ANOVA per measure on cells recorded in the box; the
box is open (large, small or both), and only the large box is in the data set. Fields are
4-connected pixels above (a) the cell's mean rate or (b) 20% of its peak rate, at least 9 pixels.
Reproduced when no genotype effect has p < 0.05 and the class effect has p < 0.05 for coherence,
active pixels, information and in-field rate.

In [ ]:
box = cells[(cells.env == "big_box") & cells.cls.isin(["place", "nonspatial", "interneuron"])].copy()
print("Large box cells by genotype and class:")
print(box.groupby(["genotype", "cls"]).size().unstack())
print("Mice:", box.groupby("genotype").mouse.nunique().to_dict())
measures = ["coherence", "active", "info", "nfields_mean", "infield_mean", "nfields_20pk", "infield_20pk"]
f2 = pd.DataFrame(anova_rows(box, measures, "large box"))
box_code = cells_code[(cells_code.env == "big_box") & cells_code.cls.isin(["place", "nonspatial", "interneuron"])].copy()
box_iso = box[box.refractory <= ISO]
f2 = pd.concat([f2, pd.DataFrame(anova_rows(box_code, measures[:-1], "32x32 grid")),
                pd.DataFrame(anova_rows(box_iso, measures, "refractory filter"))], ignore_index=True)
print(f2.round(4).to_string())
print("32x32 grid, large box cells by genotype and class:")
print(box_code.groupby(["genotype", "cls"]).size().unstack().to_string())
print("\nMeans by class and genotype:")
print(box.groupby(["cls", "genotype"])[measures].mean().round(3).to_string())
print("\nRanges:", box[measures].describe().loc[["min", "max"]].round(3).to_string())
fig, axs = plt.subplots(1, 5, figsize=(15, 3))
for ax, m in zip(axs, ["coherence", "active", "info", "nfields_mean", "infield_mean"]):
    for i, (c, g) in enumerate([(c, g) for c in ["place", "nonspatial", "interneuron"] for g in ["WT", "Fmr1-null"]]):
        v = box[(box.cls == c) & (box.genotype == g)][m].dropna()
        ax.errorbar(i, v.mean(), v.sem(), fmt="o", color="k" if g == "WT" else "r")
    ax.set_xticks(range(6)); ax.set_xticklabels(["PC WT", "PC KO", "NS WT", "NS KO", "IN WT", "IN KO"], rotation=60)
    ax.set_title(m)
plt.tight_layout(); plt.show()

**Result: not reproduced.** In the large box (213 classified clusters) the genotype effect is
significant for coherence (F = 9.01, p = 0.0030), active pixels (F = 18.7, p < 0.0001),
information (F = 7.27, p = 0.0076) and the number of fields above the mean rate (p = 0.011), with
interactions for coherence (p = 0.0077) and active pixels (p = 0.0098); the differences lie mostly
in interneurons (coherence 0.42 Fmr1-null against 0.62 WT). In-field rate (p = 0.11 and 0.42 under
the two field readings) and the count of fields above 20% of the peak (p = 0.48) show none. The
class effect is significant for every measure (p <= 0.0004). The genotype effects on coherence,
active pixels and information remain on the 32 x 32 grid (p = 0.0003, 0.0016, 0.0073) and with
the refractory filter (p = 0.0084, 0.0001, 0.013).

## tables1-place-cell-properties-similar

> Firing rate, in-field rate, spatial coherence, information content and proportion of active
> pixels of place cells, and the firing rate of interneurons, do not differ between WT and
> Fmr1-null mice.

Criterion: a t test per measure, or Mann-Whitney when a sample fails Shapiro-Wilk (Table S1
legend); reproduced when no comparison has p < 0.05. Cells and box as in fig2 (large box), in-field
rate under both field readings. Means are compared with Table S1 (tolerances its SEMs).

In [ ]:
expected = {("place", "rate"): (1.49, 0.15, 1.27, 0.12), ("place", "infield_mean"): (2.61, 0.2, 2.2, 0.13),
            ("place", "infield_20pk"): (2.61, 0.2, 2.2, 0.13), ("place", "coherence"): (0.61, 0.02, 0.57, 0.02),
            ("place", "info"): (1.57, 0.09, 1.5, 0.09), ("place", "active"): (0.45, 0.03, 0.46, 0.03),
            ("interneuron", "rate"): (16.61, 2.8, 16.02, 1.42)}
rows = []
for variant, bx in [("main", box), ("32x32 grid", box_code), ("refractory filter", box_iso)]:
  for (c, m), (ew, sw, ek, sk) in expected.items():
    a = bx[(bx.cls == c) & (bx.genotype == "WT")][m]
    b = bx[(bx.cls == c) & (bx.genotype == "Fmr1-null")][m]
    if min(a.notna().sum(), b.notna().sum()) < 3:
        continue
    r = hc16.two_group(a, b)
    rows.append(dict(variant=variant, cls=c, measure=m, n_WT=r["n_a"], WT=r["mean_a"], WT_sem=r["sem_a"], n_KO=r["n_b"], KO=r["mean_b"],
                     KO_sem=r["sem_b"], t=r["t"], p_t=r["p_t"], U=r["U"], p_U=r["p_U"], test=r["test_table"],
                     p_table=r["p_table"], d=r["d"], WT_in_tol=abs(r["mean_a"] - ew) <= sw, KO_in_tol=abs(r["mean_b"] - ek) <= sk))
s1 = pd.DataFrame(rows)
print(s1.round(4).to_string())
print("\nPlace cells by mouse (large box): n, mean rate, coherence, information")
print(box[box.cls == "place"].groupby(["genotype", "mouse"])[["rate", "coherence", "info"]].agg(["count", "mean"]).round(3).to_string())

**Result: partially reproduced.** Among large-box place cells (34 WT, 36 Fmr1-null) no measure
differs: firing rate p = 0.49, in-field rate p = 0.15 (mean-rate fields) and 0.69 (20% of peak),
coherence p = 0.39 (Mann-Whitney), information p = 0.47 (Mann-Whitney), active pixels p = 0.44.
The interneuron rates differ (WT 5.40, Fmr1-null 10.34 AP/s; Mann-Whitney p = 0.0105; with the
refractory filter p = 0.036). No mean is within Table S1's SEM: information is 2.32 and 2.27
bits/spike against 1.57 and 1.5, active pixels 0.24 and 0.26 against 0.45 and 0.46, firing rate
0.97 and 0.89 against 1.49 and 1.27 AP/s. On the 32 x 32 grid (40 and 46 place cells) no place
cell measure differs either (p = 0.12 to 0.64), and information (1.74 and 1.74 bits/spike) and
active pixels (0.40 and 0.41) come nearer Table S1 without reaching it.

## tables1-nonspatial-active-pixels

> Non-spatial pyramidal cells of Fmr1-null mice discharge in a larger proportion of the
> environment than those of WT mice (0.40 against 0.26).

Criterion: both values within tolerance (0.26 +/- 0.03, 0.40 +/- 0.05) and the t test (or
Mann-Whitney, by Shapiro-Wilk) p < 0.05 with Fmr1-null higher. Large box cells.

In [ ]:
a = box[(box.cls == "nonspatial") & (box.genotype == "WT")]["active"]
b = box[(box.cls == "nonspatial") & (box.genotype == "Fmr1-null")]["active"]
r = hc16.two_group(a, b)
print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in r.items()})
print("WT within 0.26 +/- 0.03:", abs(r["mean_a"] - 0.26) <= 0.03, "; Fmr1-null within 0.40 +/- 0.05:", abs(r["mean_b"] - 0.40) <= 0.05)
for label, t in [("R1 (all first stationary sessions)", R1), ("R2", R2), ("large box, 32x32 grid", box_code),
                 ("large box, refractory filter", box_iso)]:
    rr = hc16.two_group(t[(t.cls == "nonspatial") & (t.genotype == "WT")]["active"],
                        t[(t.cls == "nonspatial") & (t.genotype == "Fmr1-null")]["active"])
    print(f"Variant {label}: WT {rr['mean_a']:.3f} (n={rr['n_a']}), Fmr1-null {rr['mean_b']:.3f} (n={rr['n_b']}), "
          f"t p = {rr['p_t']:.4f}, U p = {rr['p_U']:.4f}, d = {rr['d']:.2f}")
print("By mouse (large box):")
print(box[box.cls == "nonspatial"].groupby(["genotype", "mouse"]).active.agg(["count", "mean"]).round(3).to_string())

**Result: not reproduced.** Large-box non-spatial pyramidal clusters discharge in 0.152 (WT,
n = 29) and 0.222 (Fmr1-null, n = 35) of the visited pixels. Fmr1-null is higher, but both samples
fail Shapiro-Wilk and Mann-Whitney gives p = 0.10 (t test p = 0.10), and neither value is within
tolerance of 0.26 and 0.40. R1 and R2 give p = 0.09 to 0.23. On the 32 x 32 grid both values are
within tolerance (0.241 and 0.355, 23 and 25 cells) but the difference is not significant
(Mann-Whitney p = 0.099); with the refractory filter p = 0.091.

## Passes and overdispersion

STAR Methods, "Overdispersion": for each 5 s interval, exp is the sum over position samples of
the time spent times the session's time-averaged rate at that pixel, z = (obs - exp)/sqrt(exp),
and the interval is a pass when exp is at least the cell's average rate. The claims file reads
this literally, a count against a rate in AP/s; the count expected at the average rate over the
5 s (exp at least the average rate x 5 s) is a variant, and which is meant is asked of a reader
(questions/crcns-hc-16.yaml). Intervals are
consecutive from the session start (the claims file's reading); a 1 s sliding step is a variant.
Place cells of the stationary sessions (large box, stationary arena 1 and 2), each session on its
own (R2); R1 (each cell once, from its first session) is a variant.

In [ ]:
def pass_table(t, step_s=None, rule="rate", index_fn=hc16.pixel_index):
    out = {}
    for sid, g in t[t.cls == "place"].groupby("session"):
        s = S[sid]
        ish = index_fn(s, "Rm")
        for c in g.cell:
            out[(sid, c)] = hc16.passes(s, c, step_s=step_s, idx_shape=ish, rule=rule)
    return out

P_R2 = pass_table(R2)
P_R2_step1 = pass_table(R2, step_s=1.0)
P_R2_count = pass_table(R2, rule="count")
pz = pd.DataFrame([dict(session=k[0], cell=k[1], genotype=S[k[0]].genotype, mouse=S[k[0]].mouse, z=zz)
                   for k, v in P_R2.items() for zz in v["z"][v["is_pass"]]])
pz["in_R1"] = pz.session.isin(set(R1.session))
pz_count = pd.DataFrame([dict(session=k[0], cell=k[1], genotype=S[k[0]].genotype, mouse=S[k[0]].mouse, z=zz)
                         for k, v in P_R2_count.items() for zz in v["z"][v["is_pass"]]])
pz_count["in_R1"] = pz_count.session.isin(set(R1.session))
for name, P in [("literal (exp >= mean rate)", P_R2), ("count (exp >= mean rate x 5 s)", P_R2_count)]:
    f = np.array([v["is_pass"].mean() for v in P.values()]); n = np.array([v["is_pass"].sum() for v in P.values()])
    print(f"Pass rule {name}: fraction of 5 s intervals that pass, median {np.median(f):.3f} (range {f.min():.3f} to {f.max():.3f}); "
          f"passes per cell median {np.median(n):.0f} (range {n.min()} to {n.max()})")
print("Place cells with passes:", pz.groupby("genotype")[["session", "cell"]].apply(lambda d: len(d.drop_duplicates())).to_dict())
print("z over passes: range %.2f to %.2f" % (pz.z.min(), pz.z.max()))

## overdispersion-passes

> Overdispersion is lower (spatial firing more reliable) in Fmr1-null mice when the unit is a
> pass through a place field (variance 4.89 against 5.81).

Criterion: both variances within tolerance (5.81 +/- 0.58, 4.89 +/- 0.49), and the F test and
Levene's test p < 0.05 with Fmr1-null lower. Pass counts expected 6,776 and 9,102 (+/- 10%).

In [ ]:
def od_passes(zz, label):
    a = zz[zz.genotype == "WT"].z.to_numpy(); b = zz[zz.genotype == "Fmr1-null"].z.to_numpy()
    F = a.var(ddof=1) / b.var(ddof=1)
    p_one = stats.f.sf(F, len(a) - 1, len(b) - 1)
    lev = stats.levene(a, b, center="mean")
    return dict(reading=label, n_WT=len(a), var_WT=a.var(ddof=1), n_KO=len(b), var_KO=b.var(ddof=1), F=F,
                p_F_one_sided=p_one, p_F_two_sided=2 * min(p_one, 1 - p_one), levene_F=lev.statistic, levene_p=lev.pvalue,
                WT_in_tol=abs(a.var(ddof=1) - 5.81) <= 0.58, KO_in_tol=abs(b.var(ddof=1) - 4.89) <= 0.49,
                passes_in_tol=(abs(len(a) - 6776) <= 678) and (abs(len(b) - 9102) <= 910))

pz1 = pd.DataFrame([dict(session=k[0], genotype=S[k[0]].genotype, z=zz)
                    for k, v in P_R2_step1.items() for zz in v["z"][v["is_pass"]]])
odp = pd.DataFrame([od_passes(pz, "R2, consecutive 5 s"), od_passes(pz[pz.in_R1], "R1, consecutive 5 s"),
                    od_passes(pz1, "R2, 5 s every 1 s"),
                    od_passes(pz_count, "R2, count rule"), od_passes(pz_count[pz_count.in_R1], "R1, count rule")])
print(odp.round(4).to_string())
print("\nBy mouse (R2): passes and variance of z")
print(pz.groupby(["genotype", "mouse"]).z.agg(["count", "var"]).round(3).to_string())
print("\nBy mouse (R2, count rule): passes and variance of z")
print(pz_count.groupby(["genotype", "mouse"]).z.agg(["count", "var"]).round(3).to_string())

**Result: not reproduced**, on the claims file's pass rule. In R2 there are 12,802 WT and 18,521
Fmr1-null passes (expected 6,776 and 9,102) with variances of z 3.69 and 3.55 (expected 5.81 and
4.89). The F test finds the WT variance larger (F = 1.04, one-sided p = 0.008) but Levene's test
does not (p = 0.998). In R1 the Fmr1-null variance is the larger (3.83 against 3.53). With 5 s
intervals every 1 s the result is as in R2 (Levene p = 0.61). The literal rule passes a median
76% of a cell's 5 s intervals. With the count rule (33% of intervals pass) R2 gives 5,442 and
8,250 passes, variances 5.28 and 4.66, both within tolerance, and Fmr1-null lower by the F test
(F = 1.13, one-sided p < 0.0001) and Levene's test (p = 0.0006), which meets the criterion; the
WT pass count is outside 6,776 +/- 678. In R1 the count rule gives Fmr1-null the larger variance
(5.16 against 5.00). Which rule the paper means is asked of a reader.

## overdispersion-cells

> Overdispersion is marginally lower in Fmr1-null mice when the unit is a place cell (5.02
> against 5.98).

Criterion: per cell, the variance of z over its passes; mean +/- SEM per genotype; minimum passes
per cell 2, 10 or 30; reproduced when under one reading both means are within tolerance
(5.98 +/- 0.44, 5.02 +/- 0.3) and the one-tailed t test gives p < 0.05, Fmr1-null lower.

In [ ]:
pc_var = pz.groupby(["session", "cell", "genotype", "in_R1"]).z.agg(["count", "var"]).reset_index()
pc_var_count = pz_count.groupby(["session", "cell", "genotype", "in_R1"]).z.agg(["count", "var"]).reset_index()
rows = []
for label, d0 in [("R2", pc_var), ("R1", pc_var[pc_var.in_R1]), ("R2, count rule", pc_var_count),
                  ("R1, count rule", pc_var_count[pc_var_count.in_R1])]:
    for mn in [2, 10, 30]:
        d = d0[d0["count"] >= mn]
        r = hc16.two_group(d[d.genotype == "WT"]["var"], d[d.genotype == "Fmr1-null"]["var"], alternative="greater")
        rows.append(dict(reading=label, min_passes=mn, n_WT=r["n_a"], WT=r["mean_a"], WT_sem=r["sem_a"], n_KO=r["n_b"],
                         KO=r["mean_b"], KO_sem=r["sem_b"], t=r["t"], p_t_one=r["p_t"], U=r["U"], p_U_one=r["p_U"], d=r["d"],
                         WT_in_tol=abs(r["mean_a"] - 5.98) <= 0.44, KO_in_tol=abs(r["mean_b"] - 5.02) <= 0.3))
odc = pd.DataFrame(rows)
print(odc.round(4).to_string())
print("Per-cell variance range: %.2f to %.2f" % (pc_var["var"].min(), pc_var["var"].max()))
print("Passes per cell, count rule: min %d, median %.0f" % (pc_var_count["count"].min(), pc_var_count["count"].median()))

**Result: not reproduced.** Every place cell has at least 30 passes, so the three minimums give
the same sample. R2: WT 3.68 +/- 0.24 (n = 55), Fmr1-null 3.53 +/- 0.21 (n = 70), one-tailed t
p = 0.32, Mann-Whitney p = 0.27; R1: 3.56 against 3.72 (p = 0.67). Neither mean is within
tolerance of 5.98 and 5.02. With the count rule (at least 49 passes per cell, again one sample)
R2 gives 5.33 +/- 0.46 against 4.69 +/- 0.33 (one-tailed t p = 0.12) and R1 5.02 against 5.08
(p = 0.54); in no reading are both means within tolerance.

## fig4b-covariance-30-passes and fig4b-covariance-40-passes

> Pairs of Fmr1-null place cells with overlapping fields covary less in their pass-by-pass
> discharge than WT pairs (average r 0.04 against 0.09), a difference that disappears after
> shuffling. / With at least 40 passes, the difference holds and is larger (0.06 against 0.19).

Criterion: for each pair of simultaneously recorded place cells, the Pearson correlation of the
two cells' z over the intervals that are passes for both, pairs with at least 30 (40) such passes;
t test on Fisher z; averages within tolerance (WT 0.09 +/- 0.03, Fmr1-null 0.04 +/- 0.01; for 40,
0.19 +/- 0.04 and 0.06 +/- 0.01); the shuffle permutes z within each cell of a pair (1 permutation
per pair; 100 as a variant).

In [ ]:
rng_cov = np.random.default_rng(404)
def pair_cov(P, nshuf=100, rng=rng_cov):
    rows = []
    by_sess = {}
    for (sid, c), v in P.items():
        by_sess.setdefault(sid, []).append((c, v))
    for sid, lst in by_sess.items():
        for i in range(len(lst)):
            for j in range(i + 1, len(lst)):
                (ca, va), (cb, vb) = lst[i], lst[j]
                both = va["is_pass"] & vb["is_pass"]
                n = int(both.sum())
                if n < 3:
                    continue
                za, zb = va["z"][both], vb["z"][both]
                r = stats.pearsonr(za, zb).statistic if za.std() > 0 and zb.std() > 0 else np.nan
                sh = [stats.pearsonr(rng.permutation(za), rng.permutation(zb)).statistic for _ in range(nshuf)]
                rows.append(dict(session=sid, genotype=S[sid].genotype, mouse=S[sid].mouse, a=ca, b=cb, n=n, r=r,
                                 r_shuf1=sh[0], r_shuf100=np.mean(hc16.fisher(np.array(sh))), r_shuf_all=np.array(sh)))
    return pd.DataFrame(rows)

cov = pair_cov(P_R2)
cov["in_R1"] = cov.session.isin(set(R1.session))
cov_count = pair_cov(P_R2_count)
cov_count["in_R1"] = cov_count.session.isin(set(R1.session))
for name, d in [("literal rule", cov), ("count rule", cov_count)]:
    print(f"Joint passes per pair, {name}: percentiles 0/10/50/90/100 =", np.percentile(d.n, [0, 10, 50, 90, 100]).round(0).tolist(),
          "; pairs with >= 30:", int((d.n >= 30).sum()), "and >= 40:", int((d.n >= 40).sum()), "of", len(d))
print("Pairs with >= 3 joint passes:", cov.groupby("genotype").size().to_dict(), "; joint passes range", cov.n.min(), "to", cov.n.max())
rows = []
for label, d0 in [("R2", cov), ("R1", cov[cov.in_R1]), ("R2, count rule", cov_count), ("R1, count rule", cov_count[cov_count.in_R1])]:
    for mn, exp in [(30, (0.09, 0.03, 0.04, 0.01)), (40, (0.19, 0.04, 0.06, 0.01))]:
        d = d0[(d0.n >= mn) & d0.r.notna()]
        a = d[d.genotype == "WT"]; b = d[d.genotype == "Fmr1-null"]
        t = stats.ttest_ind(hc16.fisher(a.r), hc16.fisher(b.r))
        u = stats.mannwhitneyu(a.r, b.r)
        t1 = stats.ttest_ind(hc16.fisher(a.r_shuf1), hc16.fisher(b.r_shuf1))
        t100 = stats.ttest_ind(a.r_shuf100, b.r_shuf100)
        rows.append(dict(reading=label, min_passes=mn, n_WT=len(a), r_WT=a.r.mean(), sem_WT=a.r.sem(), n_KO=len(b),
                         r_KO=b.r.mean(), sem_KO=b.r.sem(), t=t.statistic, p_t=t.pvalue, U=u.statistic, p_U=u.pvalue,
                         d=hc16.cohen_d(hc16.fisher(a.r), hc16.fisher(b.r)),
                         shuf1_t=t1.statistic, shuf1_p=t1.pvalue, shuf100_t=t100.statistic, shuf100_p=t100.pvalue,
                         WT_in_tol=abs(a.r.mean() - exp[0]) <= exp[1], KO_in_tol=abs(b.r.mean() - exp[2]) <= exp[3]))
cv = pd.DataFrame(rows)
print(cv.round(4).to_string())
print("\nEach of the 100 permutations as a single-permutation test (t on Fisher z of the permuted r):")
for label, d0 in [("R2", cov), ("R2, count rule", cov_count)]:
    for mn in [30, 40]:
        d = d0[(d0.n >= mn) & d0.r.notna()]
        a = np.vstack(d[d.genotype == "WT"].r_shuf_all); b = np.vstack(d[d.genotype == "Fmr1-null"].r_shuf_all)
        tt = stats.ttest_ind(hc16.fisher(a), hc16.fisher(b), axis=0)
        print(f"  {label}, >= {mn}: p < 0.05 in {int((tt.pvalue < 0.05).sum())} of 100, p < 0.01 in {int((tt.pvalue < 0.01).sum())}; "
              f"|t| median {np.median(np.abs(tt.statistic)):.2f}, max {np.abs(tt.statistic).max():.2f}; "
              f"the first permutation (shuf1 above) t = {tt.statistic[0]:.2f}, ranked {int((np.abs(tt.statistic) >= abs(tt.statistic[0])).sum())} of 100 by |t|")
print("\nBy mouse (R2, >= 30 joint passes): pairs and mean r")
print(cov[cov.n >= 30].groupby(["genotype", "mouse"]).r.agg(["count", "mean"]).round(3).to_string())
print("\nBy mouse (R2, count rule, >= 30 joint passes): pairs and mean r")
print(cov_count[cov_count.n >= 30].groupby(["genotype", "mouse"]).r.agg(["count", "mean"]).round(3).to_string())
fig, ax = plt.subplots(figsize=(4, 3))
for g, col in [("WT", "k"), ("Fmr1-null", "r")]:
    v = np.sort(cov[(cov.n >= 30) & (cov.genotype == g)].r.dropna())
    ax.plot(v, np.arange(1, len(v) + 1) / len(v), color=col, label=g)
ax.set_xlabel("pair r over joint passes (>= 30)"); ax.set_ylabel("cumulative fraction"); ax.legend()
plt.show()
print("Plotted r range: %.3f to %.3f" % (cov[cov.n >= 30].r.min(), cov[cov.n >= 30].r.max()))

**Result: partially reproduced, for both claims.** fig4b-covariance-30-passes: in R2, 122 WT and
210 Fmr1-null pairs have at least 30 joint passes (expected 109 and 249). WT r = 0.061 +/- 0.020
and Fmr1-null 0.021 +/- 0.010; t on Fisher z = 2.29, p = 0.023 (Mann-Whitney p = 0.26). WT is
within tolerance of 0.09 +/- 0.03, Fmr1-null is not within 0.04 +/- 0.01. After one permutation
per pair the difference is gone (p = 0.21; mean of 100 permutations p = 0.22). R1 gives the same
pattern (p = 0.033).

fig4b-covariance-40-passes, partially reproduced: under the literal pass rule the fewest joint
passes of any pair is 39, so the 30 and 40 pass minimums select nothing and the sample hardly
changes (122 WT, 209 Fmr1-null pairs): r = 0.061 against 0.020, t = 2.33, p = 0.021. The
difference holds but is not larger, and both averages are outside tolerance (0.19 +/- 0.04,
0.06 +/- 0.01).

With the count rule pairs have 3 to 129 joint passes (median 28), and the minimums select: at 30,
50 WT and 103 Fmr1-null pairs, r = 0.165 against 0.007, t = 3.54, p = 0.0005; at 40, 30 and 86
pairs, r = 0.279 against 0.039, t = 4.41, p < 0.0001, a larger difference, as claimed. After 100
permutations neither differs (p = 0.51, 0.96), but one permutation per pair still differs at 30
passes (p = 0.0004). That draw is a chance one: taken one at a time, the 100 permutations give
p < 0.05 in 4 of 100 at 30 passes (5 at 40), as a null should, and the first permutation, the one
reported, has the largest |t| of the 100 (3.61; median 0.68). Under the literal rule 4 and 3 of
100 give p < 0.05. No average is within tolerance under the count rule (WT above, Fmr1-null
below), so both outcomes would stay partially reproduced.

## Kendall correlations

STAR Methods, "Functional Coupling of Cell Pairs": Kendall's correlation between the spike count
series of each pair of simultaneously recorded cells, at 25, 40, 125, 250 ms, 1 s and 5 s. Tau-b,
as in the CorrelateBPFSpikeTrainEpisodes code. The claims file names the readings: K1, every bin;
K2, only bins in which the mouse is in an active pixel (rate at least the cell's mean, 32 x 32
map) of both cells; P1, the whole session; P2, the first 5 min; a pair counted once (R1 sessions)
or per session (R2). Bins are placed on the spike clock from the first position sample; the
position of a bin is the sample at its centre (hc16.Session.sample_time). Pairs are computed for
every stationary session (R2) and the periods: the session, the first 5 min, the two halves and
the second 5 min.

In [ ]:
T_K = time.time()
def session_taus(t, cls, gate, extra_periods=True):
    out = []
    for sid, g in t[t.cls == cls].groupby("session"):
        cl = list(g.cell)
        if len(cl) < 2:
            continue
        s = S[sid]
        D = s.sample_time[-1] - s.sample_time[0]
        periods = {"session": (None, None), "first5": (0, 300)}
        if extra_periods:
            periods.update({"half1": (0, D / 2), "half2": (D / 2, D), "second5": (300, 600)})
        out.append(hc16.pair_taus(s, cl, periods=periods, gate=gate))
    d = pd.concat(out, ignore_index=True)
    d["in_R1"] = d.session.isin(set(R1.session))
    d["K"] = "K2" if gate else "K1"
    return d

KT_place = pd.concat([session_taus(R2, "place", False), session_taus(R2, "place", True)], ignore_index=True)
KT_ns = pd.concat([session_taus(R2, "nonspatial", False), session_taus(R2, "nonspatial", True)], ignore_index=True)
print(f"Kendall computed in {time.time() - T_K:.0f} s")
for name, d in [("place", KT_place), ("non-spatial", KT_ns)]:
    q = d[(d.period == "session") & (d.width == 0.025) & (d.K == "K1")]
    print(f"{name} pairs (R2): {q.groupby('genotype').size().to_dict()}; in R1 sessions: {q[q.in_R1].groupby('genotype').size().to_dict()}")
    print("  tau range over all readings %.3f to %.3f; undefined tau: %d of %d" % (d.tau.min(), d.tau.max(), d.tau.isna().sum(), len(d)))
    print("  median bins entering K2 (session):", d[(d.K == "K2") & (d.period == "session")].groupby("width").nbins.median().to_dict())

def mw_table(d, expected=None):
    rows = []
    for (K, period, unit), _ in [((K, p, u), None) for K in ["K1", "K2"] for p in ["session", "first5"] for u in ["R1", "R2"]]:
        q = d[(d.K == K) & (d.period == period)]
        if unit == "R1":
            q = q[q.in_R1]
        for w in hc16.TIMESCALES:
            qq = q[q.width == w].dropna(subset=["tau"])
            a = qq[qq.genotype == "WT"].tau; b = qq[qq.genotype == "Fmr1-null"].tau
            u = stats.mannwhitneyu(a, b)
            row = dict(K=K, period=period, unit=unit, width=w, n_WT=len(a), WT=a.mean(), WT_sem=a.sem(), n_KO=len(b),
                       KO=b.mean(), KO_sem=b.sem(), U=u.statistic, p=u.pvalue, KO_lower=b.mean() < a.mean(),
                       d=hc16.cohen_d(a, b))
            if expected and w in expected:
                ew, sw, ek, sk = expected[w]
                row["WT_in_tol"] = abs(a.mean() - ew) <= sw; row["KO_in_tol"] = abs(b.mean() - ek) <= sk
            rows.append(row)
    return pd.DataFrame(rows)

## fig4cd-kendall-place-cells

> Kendall correlations between Fmr1-null place cell spike trains are weaker than between WT ones
> at the 25 ms, 40 ms and 125 ms timescales, but not significantly at 1 s and 5 s.

Criterion: Mann-Whitney between genotypes, pairs as units; reproduced when, under one reading,
25, 40 and 125 ms give p < 0.05 with Fmr1-null lower and 1 s and 5 s give p >= 0.05; partially
reproduced when two of the three short timescales do. Pair counts expected 247 and 504.

In [ ]:
exp_pc = {0.025: (0.017, 0.006, 0.009, 0.0012), 0.040: (0.02, 0.007, 0.01, 0.002), 0.125: (0.02, 0.008, 0.017, 0.003)}
kp = mw_table(KT_place, exp_pc)
print(kp.round(4).to_string())
def judge(kt, short, long_):
    out = []
    for key, g in kt.groupby(["K", "period", "unit"]):
        g = g.set_index("width")
        sig_short = [(g.loc[w, "p"] < 0.05) and g.loc[w, "KO_lower"] for w in short]
        ns_long = [g.loc[w, "p"] >= 0.05 for w in long_]
        out.append(dict(K=key[0], period=key[1], unit=key[2], short_sig=sum(sig_short), long_ns=sum(ns_long),
                        n_WT=int(g.n_WT.iloc[0]), n_KO=int(g.n_KO.iloc[0])))
    return pd.DataFrame(out)
print("\nPer reading: short timescales with p < 0.05 and Fmr1-null lower; long timescales with p >= 0.05")
print(judge(kp, [0.025, 0.040, 0.125], [1.0, 5.0]).to_string())
print("\nBy mouse (K1, session, R2): pairs and mean tau at 25 ms")
q = KT_place[(KT_place.K == "K1") & (KT_place.period == "session") & (KT_place.width == 0.025)]
print(q.groupby(["genotype", "mouse"]).tau.agg(["count", "mean"]).round(4).to_string())
print("\nWithout M16 (52 of the 122 WT pairs): per reading, and K1 over the session")
kp16 = mw_table(KT_place[KT_place.mouse != "M16"], exp_pc)
print(judge(kp16, [0.025, 0.040, 0.125], [1.0, 5.0]).to_string())
print(kp16[(kp16.K == "K1") & (kp16.period == "session")].round(4).to_string())
print("Short timescales with p < 0.05 without M16, any reading:")
print(kp16[(kp16.width <= 0.125) & (kp16.p < 0.05)].round(4).to_string())

**Result: not reproduced.** Under no reading do the 25, 40 and 125 ms comparisons reach
p < 0.05 (smallest p over the eight readings 0.24). With K1 over the session (R2, 122 WT and 210
Fmr1-null pairs, expected 247 and 504) mean tau at 25 ms is 0.029 against 0.0067, lower in
Fmr1-null, but Mann-Whitney p = 0.87, 0.67 and 0.60 at 25, 40 and 125 ms, while 1 s (p = 0.049) and
5 s (p = 0.0012) do differ, the reverse of the claimed pattern. No mean tau at 25 ms is within
tolerance under the K1 readings. Without M16, which gives 52 of the 122 WT pairs, the pattern is
the same (K1 over the session, R2: p = 0.98, 1.00 and 0.27 at 25, 40 and 125 ms; 0.025 and 0.0059
at 1 s and 5 s), so it is not one mouse's; one of the eight readings without M16 has one
short timescale below 0.05 (K1, first 5 min, R1, 125 ms, p = 0.045).

## fig4d-network-state-recurrence

> Network states recur less in Fmr1-null ensembles; the vector of pairwise Kendall correlations
> does not differ between the two halves of WT recordings but does in Fmr1-null recordings.

Criterion: for each pair, tau in the earlier and later interval; a paired t test per genotype
and timescale, and PCo, the Pearson correlation of the two vectors (all pairs of a genotype
pooled). Readings: H1 the two halves, H2 the first two 5 min intervals, each with K1 and K2.
Reproduced when, under one reading, no WT test has p < 0.05, at least five of the six Fmr1-null
tests do, and WT PCo exceeds Fmr1-null PCo at most timescales. Pairs per session (R2), as the
paper's 247 and 504 pairs; R1 shown too.

In [ ]:
rows = []
for K in ["K1", "K2"]:
    for H, (p1, p2) in [("H1", ("half1", "half2")), ("H2", ("first5", "second5"))]:
        for unit in ["R2", "R1"]:
            for g in ["WT", "Fmr1-null"]:
                for w in hc16.TIMESCALES:
                    q = KT_place[(KT_place.K == K) & (KT_place.genotype == g) & (KT_place.width == w)]
                    if unit == "R1":
                        q = q[q.in_R1]
                    a = q[q.period == p1].set_index(["session", "a", "b"]).tau
                    b = q[q.period == p2].set_index(["session", "a", "b"]).tau
                    j = pd.concat([a, b], axis=1, keys=["x", "y"]).dropna()
                    t = stats.ttest_rel(j.x, j.y)
                    rows.append(dict(K=K, H=H, unit=unit, genotype=g, width=w, n=len(j), t=t.statistic, p=t.pvalue,
                                     PCo=stats.pearsonr(j.x, j.y).statistic))
rec = pd.DataFrame(rows)
print(rec.round(4).to_string())
summ = []
for key, g in rec.groupby(["K", "H", "unit"]):
    wt = g[g.genotype == "WT"].set_index("width"); ko = g[g.genotype == "Fmr1-null"].set_index("width")
    summ.append(dict(K=key[0], H=key[1], unit=key[2], WT_sig=int((wt.p < 0.05).sum()), KO_sig=int((ko.p < 0.05).sum()),
                     PCo_WT_gt_KO=int((wt.PCo > ko.PCo).sum())))
print("\nPer reading (of 6 timescales):")
print(pd.DataFrame(summ).to_string())

**Result: not reproduced.** No reading has at least five significant Fmr1-null tests: under H1
none of the six Fmr1-null paired tests reaches p < 0.05 with K1 or K2, and under H2 at most one.
WT halves do not differ under H1; under H2 with K1 the WT first two 5 min intervals differ at 25 and
40 ms (p = 0.035, 0.016). PCo is higher in WT than in Fmr1-null at all six timescales in six of the
eight readings (K1, H1, R2, 25 ms: 0.991 against 0.827) and at five in the other two (K2, H2, R1
and R2), where Fmr1-null is the higher at 1 s (0.519 against 0.279, 0.499 against 0.266).

## kendall-nonspatial-pyramidal

> Kendall correlations between Fmr1-null non-spatial pyramidal cells are weaker than between WT
> ones at 25 ms, 40 ms, 125 ms and 1 s, but not significantly at 5 s.

Criterion: as fig4cd for non-spatial pyramidal pairs; reproduced when, under one reading, 25, 40,
125 ms and 1 s give p < 0.05 with Fmr1-null lower and 5 s gives p >= 0.05; partially reproduced
when three of the four do. Expected pairs 252 and 203.

In [ ]:
exp_ns = {0.025: (0.03, 0.006, 0.007, 0.002), 0.125: (0.04, 0.007, 0.01, 0.003)}
kn = mw_table(KT_ns, exp_ns)
print(kn.round(4).to_string())
print("\nPer reading:")
print(judge(kn, [0.025, 0.040, 0.125, 1.0], [5.0]).to_string())
print("\nBy mouse (K1, session, R2): pairs and mean tau at 25 ms")
q = KT_ns[(KT_ns.K == "K1") & (KT_ns.period == "session") & (KT_ns.width == 0.025)]
print(q.groupby(["genotype", "mouse"]).tau.agg(["count", "mean"]).round(4).to_string())

**Result: not reproduced.** There are 380 WT and 146 Fmr1-null non-spatial pairs in R2
(expected 252 and 203). With K1 over the session the distributions differ at 25, 40 and 125 ms
(p = 0.0084, 0.0024, 0.0054) but Fmr1-null mean tau is not lower (0.0099 against 0.0089 at 25 ms),
and 1 s does not differ (p = 0.12). No reading gives Fmr1-null lower at p < 0.05 on more than one
of the four timescales (K2 over the session, 25 ms, p = 0.0024 in R1).

## Refractory filter and 32 x 32 grid for the pass and Kendall claims

The two variants of the cell table (refractory filter, 32 x 32 grid; see the cell table section)
applied to the claims that use passes or Kendall correlations, R2 only. With the refractory filter
a pass or a pair enters only when its cells pass the filter. On the 32 x 32 grid the place cells
are those classified on the grid, and exp is computed from their 32 x 32 rate maps. The covariance
shuffle uses its own generator. fig4d-network-state-recurrence, kendall-quiet-states and the
rotating arena claims are not run under these variants.

In [ ]:
T_V = time.time()
keep_iso = set(zip(R2_iso.session, R2_iso.cell))
def kept(d, *cols):
    k = np.ones(len(d), bool)
    for col in cols:
        k &= np.array([(s, c) in keep_iso for s, c in zip(d.session, d[col])])
    return d[k]
def zframe(P):
    return pd.DataFrame([dict(session=k[0], cell=k[1], genotype=S[k[0]].genotype, z=zz)
                         for k, v in P.items() for zz in v["z"][v["is_pass"]]])
P_code = pass_table(R2_code, index_fn=hc16.pixel_index_code)
P_code_count = pass_table(R2_code, rule="count", index_fn=hc16.pixel_index_code)
variants = {"literal": {"as above": pz, "refractory filter": kept(pz, "cell"), "32 x 32 grid": zframe(P_code)},
            "count": {"as above": pz_count, "refractory filter": kept(pz_count, "cell"), "32 x 32 grid": zframe(P_code_count)}}
print("R2 place cells with passes:", {f"{rule}, {name}": zz.groupby("genotype")[["session", "cell"]].apply(lambda d: len(d.drop_duplicates())).to_dict()
                                    for rule, vv in variants.items() for name, zz in vv.items()})

print("\noverdispersion-passes (R2), variance of z over passes:")
print(pd.DataFrame([od_passes(zz, f"{rule} rule, {name}") for rule, vv in variants.items() for name, zz in vv.items()]).round(4).to_string())

print("\noverdispersion-cells (R2), mean of the per-cell variance of z, one-tailed WT > Fmr1-null:")
rows = []
for rule, vv in variants.items():
    for name, zz in vv.items():
        pv = zz.groupby(["session", "cell", "genotype"]).z.var().reset_index()
        r = hc16.two_group(pv[pv.genotype == "WT"].z, pv[pv.genotype == "Fmr1-null"].z, alternative="greater")
        rows.append(dict(reading=f"{rule} rule, {name}", n_WT=r["n_a"], WT=r["mean_a"], WT_sem=r["sem_a"], n_KO=r["n_b"], KO=r["mean_b"],
                         KO_sem=r["sem_b"], p_t_one=r["p_t"], p_U_one=r["p_U"],
                         WT_in_tol=abs(r["mean_a"] - 5.98) <= 0.44, KO_in_tol=abs(r["mean_b"] - 5.02) <= 0.3))
print(pd.DataFrame(rows).round(4).to_string())

print("\nfig4b-covariance-30-passes and -40-passes (R2), pair r over joint passes:")
rng_var = np.random.default_rng(406)
covs = {"literal": {"as above": cov, "refractory filter": kept(cov, "a", "b"), "32 x 32 grid": pair_cov(P_code, rng=rng_var)},
        "count": {"as above": cov_count, "refractory filter": kept(cov_count, "a", "b"), "32 x 32 grid": pair_cov(P_code_count, rng=rng_var)}}
rows = []
for rule, vv in covs.items():
    for name, d0 in vv.items():
        for mn, exp in [(30, (0.09, 0.03, 0.04, 0.01)), (40, (0.19, 0.04, 0.06, 0.01))]:
            d = d0[(d0.n >= mn) & d0.r.notna()]
            a = d[d.genotype == "WT"]; b = d[d.genotype == "Fmr1-null"]
            t = stats.ttest_ind(hc16.fisher(a.r), hc16.fisher(b.r))
            rows.append(dict(reading=f"{rule} rule, {name}", min_passes=mn, joint_min=int(d0.n.min()), n_WT=len(a), r_WT=a.r.mean(),
                             n_KO=len(b), r_KO=b.r.mean(), t=t.statistic, p_t=t.pvalue,
                             shuf1_p=stats.ttest_ind(hc16.fisher(a.r_shuf1), hc16.fisher(b.r_shuf1)).pvalue,
                             shuf100_p=stats.ttest_ind(a.r_shuf100, b.r_shuf100).pvalue,
                             WT_in_tol=abs(a.r.mean() - exp[0]) <= exp[1], KO_in_tol=abs(b.r.mean() - exp[2]) <= exp[3]))
print(pd.DataFrame(rows).round(4).to_string())

KT_place_code = pd.concat([session_taus(R2_code, "place", False), session_taus(R2_code, "place", True)], ignore_index=True)
KT_ns_code = pd.concat([session_taus(R2_code, "nonspatial", False), session_taus(R2_code, "nonspatial", True)], ignore_index=True)
for claim, kts, short, long_ in [("fig4cd-kendall-place-cells", {"refractory filter": kept(KT_place, "a", "b"), "32 x 32 grid": KT_place_code},
                                  [0.025, 0.040, 0.125], [1.0, 5.0]),
                                 ("kendall-nonspatial-pyramidal", {"refractory filter": kept(KT_ns, "a", "b"), "32 x 32 grid": KT_ns_code},
                                  [0.025, 0.040, 0.125, 1.0], [5.0])]:
    for name, kt in kts.items():
        m = mw_table(kt)
        print(f"\n{claim}, {name}: per reading, timescales {short} with p < 0.05 and Fmr1-null lower, and {long_} with p >= 0.05")
        print(judge(m, short, long_).to_string())
        print("  K1, session, R2:", "; ".join(f"{w * 1000:g} ms: WT {r.WT:.4f}, KO {r.KO:.4f}, p = {r.p:.3g}"
                                              for w, r in m[(m.K == "K1") & (m.period == "session") & (m.unit == "R2")].set_index("width").iterrows()))
print(f"\nVariants computed in {time.time() - T_V:.0f} s")

Under the claims file's literal pass rule neither variant changes these results.
overdispersion-passes: with the refractory filter the variances are 3.71 and 3.55 (Levene p =
0.86); on the 32 x 32 grid 3.73 and 3.73 (F one-sided p = 0.49); none is within tolerance.
overdispersion-cells: one-tailed p = 0.29 (filter) and 0.41 (grid), no mean within tolerance.
fig4b covariance: with the filter r = 0.061 against 0.021, p = 0.023; on the grid 183 WT and 290
Fmr1-null pairs, r = 0.074 against 0.023, p = 0.0009, where the fewest joint passes is 41, so the
30 and 40 pass minimums select the same pairs; the Fmr1-null average is outside tolerance in
every case, and on the grid the mean of 100 permutations still differs (p = 0.020) where one
permutation per pair does not (p = 0.84). fig4cd Kendall place cells: under either variant no
reading has a 25, 40 or 125 ms comparison with p < 0.05 and Fmr1-null lower (K1, session: 1 s and
5 s p = 0.043 and 0.0009 with the filter, 0.022 and 0.0011 on the grid). Kendall non-spatial
pairs: at most two of the four timescales with Fmr1-null lower at p < 0.05 (K2 over the session,
filter), none on the grid, where Fmr1-null tau is the higher (K1, session, 25 ms, 0.0110 against
0.0083).

Under the count rule, a variant asked of a reader: overdispersion-passes with the refractory
filter has both variances within tolerance (5.30, 4.66; F one-sided p < 0.0001, Levene p =
0.0004), as without it. On the grid the variances (5.34, 4.89) and the pass counts (6,712, 9,607)
are within tolerance and the F test gives p = 0.0001, but Levene's test p = 0.052.
overdispersion-cells on the grid has both means within tolerance (5.57, 5.05) with one-tailed
p = 0.18. fig4b-covariance-40-passes on the grid has 65 WT and 121 Fmr1-null pairs, r = 0.182
against 0.056, both within tolerance, t p = 0.0004, which meets its criterion; at 30 passes the
Fmr1-null average (0.026) is outside tolerance.

## kendall-quiet-states

> During quiet wakefulness and sleep, place cell pair correlations do not differ at the gamma
> timescales (25, 40 ms) but are weaker in Fmr1-null at 125 ms, 250 ms and 1 s.

Criterion: tau (K1; K2 a variant) for place cell pairs over quiet periods; the readings are the
recordings the data label as sleep or rest (there are none in the data set) and periods of speed
below 2 cm/s in the session. Reproduced when 25 and 40 ms give p >= 0.05 and 125 ms, 250 ms and 1
s give p < 0.05 with Fmr1-null lower. Speed from room-frame positions smoothed over 1 s.

In [ ]:
SPEED = {s.id: hc16.speed(s) for s in sessions}
def slow_mask(s, sample, edges):
    """Bins in which the speed is below 2 cm/s at every position sample and at the bin centre."""
    v = np.nan_to_num(SPEED[s.id], nan=np.inf)
    lo = np.searchsorted(s.sample_time, edges[:-1]); hi = np.searchsorted(s.sample_time, edges[1:])
    vmax = np.where(hi > lo, np.maximum.reduceat(v, np.clip(lo, 0, len(v) - 1)), -np.inf)
    return np.maximum(vmax, v[sample]) < 2.0
def slow_mask_centre(s, sample, edges):
    return np.nan_to_num(SPEED[s.id][sample], nan=np.inf) < 2.0
out = []
for mname, mfn in [("whole bin", slow_mask), ("centre", slow_mask_centre)]:
    for gate in [False, True]:
        for sid, g in R2[R2.cls == "place"].groupby("session"):
            cl = list(g.cell)
            if len(cl) >= 2:
                d = hc16.pair_taus(S[sid], cl, gate=gate, mask_fn=mfn)
                d["K"] = ("K2" if gate else "K1") + ("" if mname == "whole bin" else ", centre")
                out.append(d)
KQ = pd.concat(out, ignore_index=True)
KQ["in_R1"] = KQ.session.isin(set(R1.session))
slow_frac = pd.Series({sid: np.mean(np.nan_to_num(SPEED[sid], nan=np.inf) < 2) for sid in R2.session.unique()})
print("Fraction of samples below 2 cm/s per session: median %.3f, range %.3f to %.3f" % (slow_frac.median(), slow_frac.min(), slow_frac.max()))
print("Bins entering (K1, median by width), whole bin:", KQ[KQ.K == "K1"].groupby("width").nbins.median().to_dict())
print("Bins entering (K1, median by width), centre only:", KQ[KQ.K == "K1, centre"].groupby("width").nbins.median().to_dict())
rows = []
for K in ["K1", "K2", "K1, centre", "K2, centre"]:
    for unit in ["R2", "R1"]:
        q = KQ[KQ.K == K]
        if unit == "R1":
            q = q[q.in_R1]
        for w in hc16.TIMESCALES:
            qq = q[q.width == w].dropna(subset=["tau"])
            a = qq[qq.genotype == "WT"].tau; b = qq[qq.genotype == "Fmr1-null"].tau
            u = stats.mannwhitneyu(a, b)
            rows.append(dict(K=K, unit=unit, width=w, n_WT=len(a), WT=a.mean(), n_KO=len(b), KO=b.mean(), U=u.statistic,
                             p=u.pvalue, KO_lower=b.mean() < a.mean()))
kq = pd.DataFrame(rows)
print(kq.round(4).to_string())
for key, g in kq.groupby(["K", "unit"]):
    g = g.set_index("width")
    print(key, "25/40 ms p >= 0.05:", int((g.loc[[0.025, 0.040], "p"] >= 0.05).sum()), "of 2;",
          "125/250 ms/1 s p < 0.05 and Fmr1-null lower:", int(((g.loc[[0.125, 0.25, 1.0], "p"] < 0.05) & g.loc[[0.125, 0.25, 1.0], "KO_lower"]).sum()), "of 3")

**Result: not reproduced.** No recording is labelled as sleep or rest, so only the low-speed
reading runs. Bins enter when the speed is below 2 cm/s at every position sample in them (21% of
a session's samples are below 2 cm/s, median). 25 and 40 ms do not differ (K1, R2: p = 0.78,
0.45), as claimed, but neither do 125 ms, 250 ms and 1 s (p = 0.41, 0.56, 0.50); 5 s keeps a few
bins in few pairs (50 WT, 45 Fmr1-null pairs; p = 0.12). Keeping a bin when the speed at its
centre is low gives the same pattern (p = 0.39, 0.92, 0.78 at 125 ms, 250 ms and 1 s). With K2 (R2),
40 ms differs (p = 0.048) and 125 ms differs with Fmr1-null higher in mean tau (p = 0.023).

## locomotion-similar

> WT and Fmr1-null mice do not differ in locomotion speed or path linearity during the
> recordings, and the change in speed between the two halves of a session does not differ between
> genotypes.

Criterion: a two-way genotype x half ANOVA with repeated measures on half, on average speed
(Figure S5B inset); reproduced when the genotype effect and the interaction have p >= 0.05.
Recordings and mice as units, both run. Recordings: the stationary sessions used for the Kendall
claims. Linearity is not judged.

In [ ]:
import pingouin as pg
rows = []
for s in sessions:
    if s.env == "rotating_cylinder":
        continue
    v = SPEED[s.id]; h = s.nsamp // 2
    rows += [dict(session=s.id, mouse=s.mouse, genotype=s.genotype, env=s.env, half="first", speed=np.nanmean(v[:h])),
             dict(session=s.id, mouse=s.mouse, genotype=s.genotype, env=s.env, half="second", speed=np.nanmean(v[h:]))]
loc = pd.DataFrame(rows)
print("Speed (cm/s) by genotype and half, recordings:")
print(loc.groupby(["genotype", "half"]).speed.agg(["count", "mean", "sem"]).round(3).to_string())
print("Range %.2f to %.2f cm/s" % (loc.speed.min(), loc.speed.max()))
a1 = pg.mixed_anova(data=loc, dv="speed", within="half", between="genotype", subject="session")
print("\nRecordings as units:\n", a1.round(4).to_string())
lm = loc.groupby(["mouse", "genotype", "half"]).speed.mean().reset_index()
a2 = pg.mixed_anova(data=lm, dv="speed", within="half", between="genotype", subject="mouse")
print("\nMice as units:\n", a2.round(4).to_string())
print("\nBy mouse:")
print(lm.pivot_table(index=["genotype", "mouse"], columns="half", values="speed").round(2).to_string())
for env in ["big_box", "stationary_cylinder"]:
    a3 = pg.mixed_anova(data=loc[loc.env == env], dv="speed", within="half", between="genotype", subject="session")
    print(f"\nVariant, {env} recordings only:\n", a3[["Source", "F", "DF1", "DF2", [c for c in a3.columns if c.startswith("p")][0]]].round(4).to_string())
print("\nSpeed (cm/s) by environment, genotype and half, recordings; change = second - first:")
sp = loc.groupby(["env", "genotype", "half"]).speed.mean().unstack()
print(sp.assign(change=sp["second"] - sp["first"]).round(2).to_string())
for env in ["big_box", "stationary_cylinder"]:
    le = loc[loc.env == env].groupby(["mouse", "genotype", "half"]).speed.mean().reset_index()
    a4 = pg.mixed_anova(data=le, dv="speed", within="half", between="genotype", subject="mouse")
    print(f"\nVariant, {env}, mice as units ({le.mouse.nunique()} mice):\n", a4[["Source", "F", "DF1", "DF2", [c for c in a4.columns if c.startswith("p")][0]]].round(4).to_string())
lr1 = loc[loc.session.isin(set(R1.session))]
a5 = pg.mixed_anova(data=lr1, dv="speed", within="half", between="genotype", subject="session")
print(f"\nVariant, one recording per mouse and day (R1, {lr1.session.nunique()} recordings):\n", a5[["Source", "F", "DF1", "DF2", [c for c in a5.columns if c.startswith("p")][0]]].round(4).to_string())
print("\nUnits each set of recordings gives (the paper's 27 error df need 29):",
      {"stationary recordings": loc.session.nunique(), "one per mouse and day": lr1.session.nunique(),
       "large box": loc[loc.env == "big_box"].session.nunique(), "stationary arena": loc[loc.env == "stationary_cylinder"].session.nunique(),
       "with rotating sessions": loc.session.nunique() + len([s for s in sessions if s.env == "rotating_cylinder"]),
       "mice": loc.mouse.nunique()})

**Result: reproduced, resting on an assumption.** With recordings as units (34 stationary
sessions, large box and stationary arena pooled) neither the genotype (F(1,32) = 0.24, p = 0.63)
nor the interaction (F = 0.18, p = 0.67) is significant, while speed falls from the first to the
second half (F = 22.8, p < 0.0001; WT 6.63 to 5.86, Fmr1-null 6.31 to 5.66 cm/s), as the paper's
epoch effect. With mice as units: genotype p = 0.44, interaction p = 0.32, half p = 0.0058; with
one recording per mouse and day (25): interaction p = 0.20. Within one kind of environment the
interaction is significant, in opposite directions: in the large box WT slows more (by 1.37
against 0.14 cm/s; p = 0.021, 15 mice), in the stationary arena Fmr1-null slows more (1.01
against 0.25 cm/s; p = 0.0076 with recordings, 0.024 with 10 mice as units). Pooling the
environments is assumed, as the STAR Methods put all four under "Constant Conditions Open-Field"
and Figure S5B speaks of the open field recordings. No set of recordings here gives the 29 units
the paper's 27 error degrees of freedom imply. Linearity is not judged.

## Rotating arena: place cells

The claims for Figures 5 and S6 use the place cells recorded in the rotating arena. Which session
classifies them is not stated; two readings are run. A: place cells by the classification of the
rotating session itself (room frame). B: place cells by the stationary session of the same day
that shares the rotating session's cut file (stationary arena 1, or 2 for M29 whose first session
has another cut file), as the paper classifies cells in standard recordings.

In [ ]:
rot = [s for s in sessions if s.env == "rotating_cylinder"]
rot_place = {}
for s in rot:
    a = cells[(cells.session == s.id)]
    A = set(a[a.cls == "place"].cell)
    same = [x for x in sessions if x.env == "stationary_cylinder" and x.mouse == s.mouse and x.date == s.date and x.cut == s.cut]
    b = cells[cells.session == same[0].id] if same else None
    B = set(b[b.cls == "place"].cell) & set(s.cells) if b is not None else set()
    rot_place[s.id] = {"A": sorted(A), "B": sorted(B), "B_from": same[0].id if same else None}
    print(f"{s.id} ({s.genotype}): {len(s.cells)} clusters; place cells A {len(A)}, B {len(B)} (from {rot_place[s.id]['B_from']}); both {len(A & B)}")
for r in ["A", "B"]:
    print(r, {g: sum(len(rot_place[s.id][r]) for s in rot if s.genotype == g) for g in ["WT", "Fmr1-null"]}, "(paper: WT 20, Fmr1-null 31)")

# Ipos per 133 ms interval, per cluster and frame
T_I = time.time()
IPOS = {}
for s in rot:
    for c in s.cells:
        for fr in ["Rm", "Ar"]:
            IPOS[(s.id, c, fr, "abs")] = hc16.ipos_series(s, c, fr)
            IPOS[(s.id, c, fr, "signed_ln")] = hc16.ipos_series(s, c, fr, signed=True, log=np.log)
print(f"Ipos computed in {time.time() - T_I:.0f} s; intervals per session:", {s.id: len(IPOS[(s.id, s.cells[0], 'Rm', 'abs')]) for s in rot})
print("Fraction of intervals with Ipos defined (room, arena):",
      {s.id: (round(np.mean(~np.isnan(IPOS[(s.id, s.cells[0], 'Rm', 'abs')])), 3), round(np.mean(~np.isnan(IPOS[(s.id, s.cells[0], 'Ar', 'abs')])), 3)) for s in rot})

## fig5b-frame-specific-ipos

> During arena rotation, room-frame positional information (Ipos) is greater in Fmr1-null than in
> WT place cells, and arena-frame Ipos only marginally greater.

Criterion: |Ipos| per 133 ms interval from room and arena positions (32 x 32 maps, pixels with
fewer than 10 visits left out), averaged over time per cell; Mann-Whitney between genotypes;
reproduced when the room-frame comparison gives p < 0.05 with Fmr1-null higher. The code's
signed natural-log values are a variant.

In [ ]:
rows = []
for s in rot:
    for c in s.cells:
        rows.append(dict(session=s.id, genotype=s.genotype, mouse=s.mouse, cell=c,
                         A=c in rot_place[s.id]["A"], B=c in rot_place[s.id]["B"],
                         room=np.nanmean(IPOS[(s.id, c, "Rm", "abs")]), arena=np.nanmean(IPOS[(s.id, c, "Ar", "abs")]),
                         room_signed_ln=np.nanmean(IPOS[(s.id, c, "Rm", "signed_ln")]),
                         arena_signed_ln=np.nanmean(IPOS[(s.id, c, "Ar", "signed_ln")])))
ip = pd.DataFrame(rows)
rows = []
for r in ["A", "B"]:
    d = ip[ip[r]]
    for m in ["room", "arena", "room_signed_ln", "arena_signed_ln"]:
        a = d[d.genotype == "WT"][m]; b = d[d.genotype == "Fmr1-null"][m]
        u = stats.mannwhitneyu(a, b)
        rows.append(dict(reading=r, measure=m, n_WT=len(a), WT=a.mean(), WT_sem=a.sem(), n_KO=len(b), KO=b.mean(),
                         KO_sem=b.sem(), U=u.statistic, p=u.pvalue, KO_higher=b.mean() > a.mean(), d=hc16.cohen_d(b, a)))
f5b = pd.DataFrame(rows)
print(f5b.round(4).to_string())
print("\nBy mouse (reading A): place cells, mean room and arena Ipos")
print(ip[ip.A].groupby(["genotype", "mouse"])[["room", "arena"]].agg(["count", "mean"]).round(4).to_string())
print("Room minus arena Ipos, place cells A: WT %.4f, Fmr1-null %.4f; cells with room > arena: %d of %d"
      % ((ip[ip.A & (ip.genotype == 'WT')].room - ip[ip.A & (ip.genotype == 'WT')].arena).mean(),
         (ip[ip.A & (ip.genotype == 'Fmr1-null')].room - ip[ip.A & (ip.genotype == 'Fmr1-null')].arena).mean(),
         int((ip[ip.A].room > ip[ip.A].arena).sum()), int(ip.A.sum())))
print("\nPer cell, mean |Ipos| (bits) in the room and arena frame, place cells of reading A or B:")
print(ip[ip.A | ip.B][["genotype", "mouse", "session", "cell", "A", "B", "room", "arena"]].sort_values(["genotype", "room"]).round(4).to_string())
print("\nReading B, room frame, leaving out one mouse at a time: Mann-Whitney p")
d = ip[ip.B]
print({m: round(stats.mannwhitneyu(d[(d.mouse != m) & (d.genotype == "WT")].room, d[(d.mouse != m) & (d.genotype == "Fmr1-null")].room).pvalue, 4)
       for m in sorted(d.mouse.unique())})

**Result: partially reproduced.** Room-frame Ipos is higher in Fmr1-null under both readings,
but the difference is significant in one, and rests on one mouse. With place cells classified in
the stationary session of the same day (reading B; 10 WT, 8 Fmr1-null) room-frame Ipos is greater
in Fmr1-null (0.106 against 0.066 bits, U = 12, p = 0.012); arena-frame Ipos is greater too (0.097
against 0.063, p = 0.016), where the paper finds it only marginal. Leaving out M37 (3 of the 8
Fmr1-null cells) gives p = 0.13; leaving out any other mouse keeps p below 0.05. With place
cells classified in the rotating session itself (reading A; 9 and 7) the room-frame difference
has the same direction but is not significant (0.103 against 0.084, p = 0.25). The samples come
from 3 mice per genotype and are about half the paper's 20 and 31 cells.

## fig5c-ensemble-coordination

> The frame-specific positional discharge fluctuations of single place cells are less correlated
> with those of the rest of the ensemble in Fmr1-null than in WT mice.

Criterion: per cell, delta Ipos = Ipos(room) - Ipos(arena) per 133 ms interval; the ensemble
series from the frame-specific sums of Ipos over the other simultaneously recorded cells; the
Pearson correlation of the cell's and the ensemble's series; delta and |delta| both run; t test on
Fisher z; reproduced when, under one reading, p < 0.05 with Fmr1-null lower. The ensemble is
"all simultaneously recorded cells except one" (Results); the other place cells only is a
variant. Place cell readings A and B.

In [ ]:
rows = []
for s in rot:
    allc = s.cells
    room = np.vstack([IPOS[(s.id, c, "Rm", "abs")] for c in allc])
    arena = np.vstack([IPOS[(s.id, c, "Ar", "abs")] for c in allc])
    for r in ["A", "B"]:
        pcs = rot_place[s.id][r]
        for c in pcs:
            i = allc.index(c)
            for ens_name, members in [("all clusters", [k for k in range(len(allc)) if k != i]),
                                      ("place cells", [allc.index(x) for x in pcs if x != c])]:
                if len(members) == 0:
                    continue
                d_cell = room[i] - arena[i]
                d_ens = np.nansum(room[members], 0) - np.nansum(arena[members], 0)
                ok = ~np.isnan(d_cell) & ~np.isnan(room[members]).all(0) & ~np.isnan(arena[members]).all(0)
                for form in ["delta", "abs delta"]:
                    x = d_cell[ok]; y = d_ens[ok]
                    if form == "abs delta":
                        x, y = np.abs(x), np.abs(y)
                    rr = stats.pearsonr(x, y).statistic if x.std() > 0 and y.std() > 0 else np.nan
                    rows.append(dict(session=s.id, genotype=s.genotype, mouse=s.mouse, cell=c, reading=r, ensemble=ens_name,
                                     form=form, n_ens=len(members), r=rr))
ec = pd.DataFrame(rows)
rows = []
for key, g in ec.groupby(["reading", "ensemble", "form"]):
    a = g[g.genotype == "WT"].r.dropna(); b = g[g.genotype == "Fmr1-null"].r.dropna()
    if len(a) < 2 or len(b) < 2:
        continue
    t = stats.ttest_ind(hc16.fisher(a), hc16.fisher(b))
    rows.append(dict(reading=key[0], ensemble=key[1], form=key[2], n_WT=len(a), r_WT=a.mean(), sem_WT=a.sem(),
                     n_KO=len(b), r_KO=b.mean(), sem_KO=b.sem(), t=t.statistic, p=t.pvalue, KO_lower=b.mean() < a.mean(),
                     d=hc16.cohen_d(hc16.fisher(a), hc16.fisher(b))))
f5c = pd.DataFrame(rows)
print(f5c.round(4).to_string())
print("\nBy mouse (A, all clusters, abs delta):")
print(ec[(ec.reading == "A") & (ec.ensemble == "all clusters") & (ec.form == "abs delta")].groupby(["genotype", "mouse"]).r.agg(["count", "mean"]).round(3).to_string())
print("r range: %.3f to %.3f" % (ec.r.min(), ec.r.max()))

**Result: not reproduced.** Under every reading the Fmr1-null place cells correlate with the
ensemble at least as well as the WT cells. With |delta Ipos| and all other clusters as the
ensemble, r = 0.071 (Fmr1-null) against 0.070 (WT) in reading A (p = 0.98) and 0.076 against
0.045 in reading B (p = 0.27); with delta Ipos, Fmr1-null is higher in reading B with all
clusters (0.175 against 0.115, p = 0.045) and with the other place cells (0.092 against 0.045,
p = 0.0042).

## figs6-room-frame-preference

> Session-averaged spatial firing during rotation is better organized in the room frame than in
> the arena frame, similarly in both genotypes.

Criterion: a two-way genotype x environment ANOVA with repeated measures on environment (Stationary
I, Stationary II, rotating room frame, rotating arena frame) on coherence and information
content (log transformed; the reverse as a variant), with Tukey post hoc tests; reproduced when
the environment effect has p < 0.05 for coherence with the rotating arena frame below both
stationary sessions, and the interaction has p >= 0.05. Cells must be present in all four maps,
so they must share the cut file across the day's three sessions (M29, whose stationary arena 1
has another cut file, drops out). Tukey's HSD is applied across the four environments without the
repeated structure (statsmodels); paired t tests are shown as a variant.

In [ ]:
import pingouin as pg
from statsmodels.stats.multicomp import pairwise_tukeyhsd
def map_measures(s, c, frame):
    idx, shape = hc16.pixel_index(s, frame)
    rm, o = hc16.rate_map(s, c, idx, shape)
    return hc16.coherence(rm), hc16.information(rm, o)

rows = []
for s in rot:
    st = {x.cond: x for x in sessions if x.env == "stationary_cylinder" and x.mouse == s.mouse and x.date == s.date and x.cut == s.cut}
    if not {"stationary1", "stationary2"} <= set(st):
        print(f"{s.id}: no stationary arena 1 and 2 with the same cut file; left out")
        continue
    for r in ["A", "B"]:
        for c in rot_place[s.id][r]:
            if c not in st["stationary1"].cells or c not in st["stationary2"].cells:
                continue
            for env, (ss, fr) in {"Stationary I": (st["stationary1"], "Rm"), "Stationary II": (st["stationary2"], "Rm"),
                                  "Rotating room": (s, "Rm"), "Rotating arena": (s, "Ar")}.items():
                coh, inf = map_measures(ss, c, fr)
                rows.append(dict(reading=r, session=s.id, genotype=s.genotype, mouse=s.mouse, cell=f"{s.id}.{c}", env=env,
                                 coherence=coh, info=inf, log_info=np.log(inf), log_coherence=np.log(coh) if coh > 0 else np.nan))
fs6 = pd.DataFrame(rows)
for r in ["A", "B"]:
    d = fs6[fs6.reading == r].dropna(subset=["coherence", "log_info"])
    full = d.groupby("cell").env.nunique() == 4
    d = d[d.cell.isin(full[full].index)]
    print(f"\nReading {r}: cells with all four maps: {d.groupby('genotype').cell.nunique().to_dict()} (paper: WT 20, Fmr1-null 31)")
    print(d.groupby(["env", "genotype"])[["coherence", "info"]].mean().round(3).unstack().to_string())
    for m in ["coherence", "log_info", "info", "log_coherence"]:
        dd = d.dropna(subset=[m])
        keep = dd.groupby("cell").env.nunique() == 4
        dd = dd[dd.cell.isin(keep[keep].index)]
        a = pg.mixed_anova(data=dd, dv=m, within="env", between="genotype", subject="cell")
        pcol = [c for c in a.columns if c.startswith("p")][0]
        print(f"  {m}: " + "; ".join(f"{row.Source} F({row.DF1},{row.DF2}) = {row.F:.2f}, p = {row[pcol]:.4g}" for _, row in a.iterrows()))
        if m in ["coherence", "log_info"]:
            tk = pairwise_tukeyhsd(dd[m], dd.env)
            print("  Tukey HSD:\n" + "\n".join("    " + l for l in str(tk.summary()).splitlines()[2:]))
            w = dd.pivot(index="cell", columns="env", values=m)
            for e in ["Stationary I", "Stationary II"]:
                tt = stats.ttest_rel(w["Rotating arena"], w[e])
                print(f"    paired t, Rotating arena vs {e}: mean diff {np.mean(w['Rotating arena'] - w[e]):.3f}, t = {tt.statistic:.2f}, p = {tt.pvalue:.4g}")

**Result: partially reproduced, resting on an assumption.** In reading B (10 WT and 7
Fmr1-null place cells with all four maps) coherence has an environment effect (F(3,45) = 6.24,
p = 0.0012) and no interaction (p = 0.58); Tukey's test puts the rotating arena frame below
stationary arena I (difference 0.29, adjusted p = 0.011) but not below stationary arena II (0.12,
p = 0.53; paired t p = 0.056). Log information has an environment effect (p = 0.0009) and no
interaction (p = 0.13). In reading A (9 and 6 cells) the coherence environment effect is not
significant (p = 0.067). The outcome adopts reading B, an assumption. It settles a question the
record leaves open elsewhere: fig5b-frame-specific-ipos is judged on both readings, and the
question is put to the paper's authors. Under reading A this claim would not be reproduced.

## Summary

| Claim | Outcome |
|---|---|
| fig1ab-baseline-transmission | blocked: data missing (slice recordings) |
| fig1cd-potentiation | blocked: data missing (slice recordings) |
| figs1-avoidance-equal | blocked: data missing (slice-cohort behaviour) |
| unit-and-cell-counts | blocked: data missing (15 of 21 mice; no isolation measures) |
| interneuron-proportion | partially reproduced (direction holds; percentages far higher) |
| place-cell-proportion | not reproduced |
| figs3-nonspatial-properties-similar | partially reproduced (rate differs) |
| fig2-spatial-quality-similar | not reproduced (genotype effects, mostly in interneurons) |
| tables1-place-cell-properties-similar | partially reproduced (place cells alike; interneuron rate differs) |
| tables1-nonspatial-active-pixels | not reproduced |
| figs4-remapping-similar | blocked: data missing (no square and circular recordings of the same cells) |
| fig3b-place-cell-modulation-similar, fig3bc-weaker-modulation-nonspatial-interneurons, fig3b-phase-preference, fig3d-stereotypy, figs5a-power-spectra-similar | blocked: data missing (no LFP) |
| overdispersion-passes | not reproduced (literal pass rule; the count rule meets the criterion in R2, asked of a reader) |
| overdispersion-cells | not reproduced |
| fig4b-covariance-30-passes | partially reproduced |
| fig4b-covariance-40-passes | partially reproduced |
| fig4cd-kendall-place-cells | not reproduced |
| fig4d-network-state-recurrence | not reproduced |
| kendall-nonspatial-pyramidal | not reproduced |
| kendall-quiet-states | not reproduced |
| locomotion-similar | reproduced (assumption: environments pooled) |
| fig5b-frame-specific-ipos | partially reproduced (significant in one reading, resting on one mouse) |
| fig5c-ensemble-coordination | not reproduced |
| figs6-room-frame-preference | partially reproduced (assumption: cells classified in the stationary session) |
| fig6-covariance-sem-predicts-errors, fig6-kendall-predicts-errors, discussion-single-cell-measures-no-relation, fig6-avoidance-behavior | blocked: data missing (avoidance sessions) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's:

- The paper's cells are units with IsoIBG and IsoINN of at least 4 bits. The data set has no
  isolation measures and no waveforms, so every sorted cluster other than cluster 0 is used, and
  cells are classified by rate and bursting without waveform width. Table S2 lists fewer cells per
  session than the data set has clusters (for example 15 cells for M16 in stationary arena 1,
  against 50 clusters here), so the population here includes clusters the paper excluded. This
  bears on every tested claim. A refractory filter, a weak proxy for isolation, removes 70 of 774
  clusters. Among the single-cell claims it removes the rate effect of
  figs3-nonspatial-properties-similar and, in R1, the significance of the interneuron difference,
  and leaves the other results as they are. Run on the pass and Kendall claims (R2), it leaves
  their results as they are. It was not run on fig4d-network-state-recurrence,
  kendall-quiet-states, locomotion-similar or the rotating arena claims.
- The data set holds 8 WT and 7 Fmr1-null mice, the large box and the stationary and rotating
  arena. The small box, the ring, sleep recordings, LFP, slice data and avoidance sessions are not
  in it.
- The spatial scale is assumed from the tracked extent and the enclosure sizes of the Methods. On
  the code's 32 x 32 grid, which needs no scale, 45 of 774 clusters change class, and the outcome
  of no single-cell claim changes. On the grid, under the literal pass rule, the results of the
  pass and Kendall claims (R2) are as on 1.25 cm pixels; under the count rule
  fig4b-covariance-40-passes meets its criterion. The grid was not run on
  fig4d-network-state-recurrence, kendall-quiet-states, locomotion-similar or the rotating arena
  claims (there Ipos is computed on the 32 x 32 grid, but the place cells are classified on
  1.25 cm pixels).
- The pass rule of the overdispersion claims is read literally, as the claims file does; it
  passes most 5 s intervals. The count expected at the mean rate over 5 s, a variant, changes the
  overdispersion-passes result in R2 (not in R1, and not overdispersion-cells), and with the
  32 x 32 grid the fig4b-covariance-40-passes result; it is asked of a reader.
- The session that classifies a cell is open (R1, R2; A, B for the rotating arena); the results
  under each are printed. A cluster name within a day's cut file is taken to be one cell. R2
  counts the cells and pairs of stationary arena 1 and 2 of a day twice; R1 counts them once.

What was found about the data set:

- It holds part of the paper's data: 15 of 21 mice, three of the four environments, no LFP, no
  sleep or rest recordings, no avoidance sessions, and no slice recordings.
- No isolation measures, waveforms or cell classes are given, so the paper's cells cannot be
  selected; the files hold every cluster, including cluster 0 (unsorted spikes). The TS headers
  hold only the sampling rate, the first timestamp and a track-to-map scale of 1.00.
- The descriptor gives no spatial scale for the 0-255 pixel positions, and the scale differs
  between box sessions (about 3.6 and 4.4 px/cm by extent).
- The stated position sampling rate drifts against the spike clock (spikes up to 118 samples, about
  4 s, after the nominal time of their position sample); spikes are filed under the right sample,
  so the files are consistent, but the nominal sample times are not usable as times.
- Six spike trains have one backward step in their times (up to 1.4 s).
- The 0-15 and 15-30 min files repeat the full sessions except one position sample at the split,
  which is in neither part, so a spike filed under it is lost; the `.times` files repeat the TS
  spike times.
- The descriptor's example header gives 58.3 samples/s; every file has 30.0.


In [ ]:
print(f"Notebook run time: {(time.time() - T0) / 60:.1f} min")